# ThingsBoard Full Harvest v11 — All Discovered Keys

## What's new vs v9 (from Cell 12 key discovery):

| New Key Group | Keys Added | Coverage in v9 |
|---|---|---|
| Dahua NVR | `Dahua_NVR_cameraInfo`, `Dahua_NVR_*` | 36% |
| Integrated Alarm | `integratedStatus`, `integratedType` | 32% |
| Tailscale VPN | `tailscale_hostname`, `tailscale_ip` | 35% |
| SW OTA metadata | `sw_id`, `sw_tag`, `sw_title`, `sw_size`, `sw_checksum`, `sw_checksum_algorithm` | 35% |
| Access Control events | `ACCESS CONTROL SYSTEM TAMPER RESTORED`, `accessControlCreatedTime` | 26–51% |
| Mili timestamps | `gateMiliTime`, `cctvMiliTime`, `timeLockMiliTime` | 32% |
| Misc new | `res`, `error`, `sw_id` | 42%, 32% |
| Scoring | `integratedStatus` now scored like IAS/FAS | ✅ |

## All 12 Bank Hierarchies (unchanged from v9):
```
Bank of India        : Tenant → Customer → HO → NBG/FGMO → ZO → Branch
Bank of Baroda       : Tenant → Customer → HO → ZO → RO → Branch
Canara Bank          : Tenant → Customer → HO → RO → Branch
Bank of Maharashtra  : Tenant → Customer → HO → ZO → Branch
Central Bank of India: Tenant → Customer → Corporate Office → ZO → RO → Branch
Indian Bank          : Tenant → Customer → HO → ZO → Branch
Indian Overseas Bank : Tenant → Customer → HO → RO → Branch
Punjab & Sind Bank   : Tenant → Customer → HO → ZO → Branch
Punjab National Bank : Tenant → Customer → HO → ZO → CO → Branch
State Bank of India  : Tenant → Customer → HO → LHO → ZO → RBO → Branch
UCO Bank             : Tenant → Customer → HO → ZO → Branch
Union Bank of India  : Tenant → Customer → Central Office → ZO → RO → Branch
```

> **Run cells 1–14 top to bottom. Set `.env` before Cell 3.**

---
## Cell 1 — Environment Setup

In [1]:
import pathlib
ENV = pathlib.Path('.env')
if not ENV.exists():
    ENV.write_text(
        'TB_HOST=https://seple.iot-private.cloud\n'
        'TB_EMAIL=info@seple.in\n'
        'TB_PASSWORD=yourpassword\n'
    )
    print('✅ .env created — fill TB_PASSWORD before continuing.')
else:
    print('✅ .env exists.')
print('   Add .env to .gitignore — never commit secrets.')


✅ .env exists.
   Add .env to .gitignore — never commit secrets.


---
## Cell 2 — Install Dependencies

In [2]:
import subprocess, sys
for p in ['requests','pandas','openpyxl','tqdm','urllib3','python-dotenv']:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])
print('✅ All packages ready.')


✅ All packages ready.


---
## Cell 3 — Config + Auth + All Key Definitions

In [3]:
import requests, json, time, warnings, os, re
from datetime import datetime
from collections import Counter, defaultdict
from dotenv import load_dotenv

load_dotenv(override=True)
warnings.filterwarnings('ignore')

TB_HOST     = os.environ['TB_HOST']
TB_EMAIL    = os.environ['TB_EMAIL']
TB_PASSWORD = os.environ['TB_PASSWORD']

PAGE_SIZE          = 100
REQUEST_DELAY      = 0.05
MAX_RELATION_DEPTH = 8
GAP_FAULT_DAYS     = 3

# ══════════════════════════════════════════════════════════════════════════════
# PER-BANK HIERARCHY MAP (unchanged from v9)
# ══════════════════════════════════════════════════════════════════════════════
BANK_HIERARCHY = {
    'BANK OF INDIA': {
        'depth': 5, 'levels': ['ho','nbg','zo','branch'],
        'type_map': {'Head Office BOI':'ho','NBG BOI':'nbg','Zonal Office BOI':'zo',
                     'Branch BOI':'branch','HO':'ho','NBG':'nbg','FGMO':'nbg','ZO':'zo'},
    },
    'BANK OF BARODA': {
        'depth': 5, 'levels': ['ho','zo','ro','branch'],
        'type_map': {'Head Office BOB':'ho','Zonal Office BOB':'zo','Regional Office BOB':'ro',
                     'Branch BOB':'branch','HO':'ho','ZO':'zo','RO':'ro'},
    },
    'CANARA BANK': {
        'depth': 4, 'levels': ['ho','ro','branch'],
        'type_map': {'Head Office CB':'ho','Regional Office CB':'ro','Branch CB':'branch',
                     'HO':'ho','RO':'ro'},
    },
    'BANK OF MAHARASHTRA': {
        'depth': 4, 'levels': ['ho','zo','branch'],
        'type_map': {'Bank-Head Office':'ho','Bank-Zonal Office':'zo','Bank-Branch':'branch',
                     'HO':'ho','ZO':'zo'},
    },
    'CENTRAL BANK OF INDIA': {
        'depth': 5, 'levels': ['ho','zo','ro','branch'],
        'type_map': {'Corporate Office':'ho','ZO':'zo','RO':'ro','Bank-Branch':'branch'},
    },
    'INDIAN BANK': {
        'depth': 4, 'levels': ['ho','zo','branch'],
        'type_map': {'HO':'ho','ZO':'zo','Branch':'branch'},
    },
    'INDIAN OVERSEAS BANK': {
        'depth': 4, 'levels': ['ho','ro','branch'],
        'type_map': {'HO':'ho','RO':'ro','Branch':'branch'},
    },
    'PANJAB & SIND BANK': {
        'depth': 4, 'levels': ['ho','zo','branch'],
        'type_map': {'HO':'ho','ZO':'zo','Branch':'branch'},
    },
    'PUNJAB NATIONAL BANK': {
        'depth': 5, 'levels': ['ho','zo','co','branch'],
        'type_map': {'HO':'ho','ZO':'zo','CO':'co','Circle Office':'co','Branch':'branch'},
    },
    'STATE BANK OF INDIA': {
        'depth': 6, 'aliases': ['SBI','STATE BANK'],
        'levels': ['ho','lho','zo','rbo','branch'],
        'type_map': {'HO':'ho','LHO':'lho','Local Head Office':'lho','SBI LHO':'lho',
                     'ZONE':'zo','ZO':'zo','RBO':'rbo','Branch':'branch'},
    },
    'UCO BANK': {
        'depth': 4, 'levels': ['ho','zo','branch'],
        'type_map': {'HO':'ho','ZO':'zo','Branch':'branch'},
    },
    'UNION BANK OF INDIA': {
        'depth': 5, 'levels': ['ho','zo','ro','branch'],
        'type_map': {'Central Office':'ho','ZO':'zo','RO':'ro','Branch':'branch'},
    },
}

GENERIC_LEVEL_MAP = {
    'Head Office BOI':'ho','Head Office BOB':'ho','Head Office CB':'ho',
    'Bank-Head Office':'ho','Corporate Office':'ho','Central Office':'ho',
    'Demo HO':'ho','Head Office':'ho','HO':'ho',
    'NBG BOI':'nbg','Demo NBG':'nbg','NBG':'nbg','FGMO':'nbg',
    'LHO':'lho','Local Head Office':'lho',
    'Zonal Office BOI':'zo','Zonal Office BOB':'zo','Bank-Zonal Office':'zo',
    'Demo ZO':'zo','ZO':'zo','Zonal Office':'zo','zo':'zo',
    'Regional Office BOB':'ro','Regional Office CB':'ro',
    'RO':'ro','Regional Office':'ro','ro':'ro',
    'CO':'co','Circle Office':'co','Circle':'co',
    'RBO':'rbo','Regional Banking Office':'rbo',
    'Branch BOI':'branch','Branch BOB':'branch','Branch CB':'branch',
    'Bank-Branch':'branch','Demo Branch':'branch',
    'Branch':'branch','branch':'branch','Site':'branch','Location':'branch',
    # Event-flag pseudo-assets → ignore
    'POWER OFF':'_ignore','MAINS ON':'_ignore','NETWORK':'_ignore',
    'DVR/NVR OFF':'_ignore','BATTERY LOW':'_ignore','SYSTEM ON':'_ignore',
    'FIRE ALARM SYSTEM OFF':'_ignore','FIRE ALARM SYSTEM ON':'_ignore',
    'INTRUSION ALARM SYSTEM OFF':'_ignore','INTRUSION ALARM SYSTEM ON':'_ignore',
    'TIME LOCK SYSTEM ON':'_ignore','HDD ERROR RESTORED':'_ignore',
    'CAMERA CONNECTION ESTABLISHED':'_ignore',
    'CAMERA TAMPERED RESTORED CH 1':'_ignore','CAMERA TAMPERED RESTORED CH 8':'_ignore',
    'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'INTRUSION ALARM SYSTEM FAULT':'_ignore',
    'INTRUSION ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'FIRE ALARM SYSTEM ACTIVATION RESTORED':'_ignore',
    'CAMERA CONNECTION ESTABLISHED CH 2':'_ignore',
    'Restricted':'_ignore','Unloading':'_ignore','Loading':'_ignore',
    'Mine site':'_ignore','default':'_ignore',
    'Regional Office BOI':'ro','Bank-Regional Office':'ro',
}

def get_asset_level(asset_type, bank_name=''):
    for bank_key, bank_cfg in BANK_HIERARCHY.items():
        if bank_key.lower() in (bank_name or '').lower():
            if asset_type in bank_cfg['type_map']:
                return bank_cfg['type_map'][asset_type]
    if asset_type in GENERIC_LEVEL_MAP:
        return GENERIC_LEVEL_MAP[asset_type]
    alow = asset_type.lower()
    for k, v in GENERIC_LEVEL_MAP.items():
        if k.lower() in alow and v != '_ignore':
            return v
    return 'other'

_LEVEL_KW = [
    (re.compile(r'\bLHO\b|Local\s+Head\s+Office', re.I), 'lho'),
    (re.compile(r'\bRBO\b|Regional\s+Banking', re.I),    'rbo'),
    (re.compile(r'\bNBG\b|\bFGMO\b', re.I),             'nbg'),
    (re.compile(r'\bZO\b|\bZONE\b|Zonal\s+Office', re.I),'zo'),
    (re.compile(r'\bRO\b|Regional\s+Office', re.I),      'ro'),
    (re.compile(r'\bCO\b|Circle\s+Office', re.I),        'co'),
    (re.compile(r'\bHO\b|Head\s+Office|Corporate\s+Office|Central\s+Office', re.I), 'ho'),
]
def classify_entity_level(name):
    for pat, level in _LEVEL_KW:
        if pat.search(name or ''): return level
    return None

# ══════════════════════════════════════════════════════════════════════════════
# CLIENT ATTRIBUTE KEYS
# ══════════════════════════════════════════════════════════════════════════════
CLIENT_KEYS = [
    # Hikvision NVR (existing)
    'dexter_config','Hikvision_NVR_cameraInfo','Hikvision_NVR_deviceID',
    'Hikvision_NVR_deviceName','Hikvision_NVR_deviceType',
    'Hikvision_NVR_firmwareVersion','Hikvision_NVR_hardwareVersion',
    'Hikvision_NVR_HDDInfo','Hikvision_NVR_macAddress',
    'Hikvision_NVR_Manufacturer','Hikvision_NVR_model',
    'Hikvision_NVR_Processor','Hikvision_NVR_serialNumber',
    # ── NEW v11: Dahua NVR (36% coverage in v9) ───────────────────────────────
    'Dahua_NVR_cameraInfo','Dahua_NVR_deviceID','Dahua_NVR_deviceName',
    'Dahua_NVR_deviceType','Dahua_NVR_firmwareVersion','Dahua_NVR_hardwareVersion',
    'Dahua_NVR_HDDInfo','Dahua_NVR_macAddress','Dahua_NVR_Manufacturer',
    'Dahua_NVR_model','Dahua_NVR_Processor','Dahua_NVR_serialNumber',
    # ── NEW v11: Tailscale VPN (35% coverage) ─────────────────────────────────
    'tailscale_hostname','tailscale_ip',
    # ── NEW v11: SW OTA metadata (35% coverage) ───────────────────────────────
    'sw_id','sw_tag','sw_title','sw_size','sw_checksum','sw_checksum_algorithm',
    # Existing
    'lastUpdate','unknown',
]

# ══════════════════════════════════════════════════════════════════════════════
# SERVER ATTRIBUTE KEYS (v9 + all new keys from v9 Cell 12 discovery)
# ══════════════════════════════════════════════════════════════════════════════
SERVER_KEYS = [
    'accessControl','accessControlDoor','accessControlHealth','accessControlStatus',
    'acsDoorOpen_history','acsOff_history','acsTamper_history',
    # ── NEW v11: ACS events (26–51% coverage) ─────────────────────────────────
    'ACCESS CONTROL SYSTEM TAMPER RESTORED','accessControlCreatedTime',
    'active','alarm','alarmFlag',
    'bas','basAlarmCreatedTime','basFault_history','basHealth','basOff_history',
    'basStatus','basSystem',
    'BATTERY LOW','BATTERY REVERSE','BATTERY ON',
    'branch_id','branchName',
    'CAMERA CONNECTION ESTABLISHED','CAMERA DISCONNECT',
    'CAMERA TAMPER','CAMERA TAMPERED RESTORED',
    'cameraDisconnectCH1_history','cameraDisconnectCH2_history',
    'cameraDisconnectCH3_history','cameraDisconnectCH4_history',
    'cameraDisconnectCH5_history','cameraDisconnectCH6_history',
    'cameraDisconnectCH7_history','cameraDisconnectCH8_history',
    'cameraDisconnectCH9_history','cameraDisconnectCH10_history',
    'cameraDisconnectCH11_history','cameraDisconnectCH12_history',
    'cameraDisconnectCH13_history','cameraDisconnectCH14_history',
    'cameraDisconnectCH15_history','cameraDisconnectCH16_history',
    'cameraDisconnectCount','cameraLinkStatus','cameraStatus',
    'cameraTamperCH1_history','cameraTamperCH2_history',
    'cameraTamperCH3_history','cameraTamperCH4_history',
    'cameraTamperCH5_history','cameraTamperCH6_history',
    'cameraTamperCH7_history','cameraTamperCH8_history',
    'cameraTamperCH9_history','cameraTamperCH10_history',
    'cameraTamperCH11_history','cameraTamperCH12_history',
    'cameraTamperCH13_history','cameraTamperCH14_history',
    'cameraTamperCH15_history','cameraTamperCH16_history',
    'cameraTamperCount','care','cctv','cctvAlarmCreatedTime','cctvStatus',
    # ── NEW v11: cctv mili time ───────────────────────────────────────────────
    'cctvMiliTime',
    'count_CH','count_HDD','critical','deviceName','deviceType',
    'DVR/NVR OFF','DVR/NVR ON','dvrNvrOff_history','eventMetadata',
    # ── NEW v11: generic error/result fields ──────────────────────────────────
    'error','res',
    'fas','fasAlarmCreatedTime','fasf','fasFault_history','fasHealth',
    'fasOff_history','fasStatus','fasSystem',
    'Faulty Device(Intrusion)','Faulty Device(Time Lock)',
    'FIRE ALARM SYSTEM ACTIVATE','FIRE ALARM SYSTEM ACTIVATION RESTORED',
    'FIRE ALARM SYSTEM ACTIVE','FIRE ALARM SYSTEM FAULT',
    'FIRE ALARM SYSTEM FAULT CONDITION RESTORED',
    'FIRE ALARM SYSTEM OFF','FIRE ALARM SYSTEM ON',
    'fireAlarmStatus','fireAlarmType','formattedBranchName',
    'gateway','gatewayAlarmCreatedTime','gatewayStatus','gatewayType',
    # ── NEW v11: gate mili time ───────────────────────────────────────────────
    'gateMiliTime',
    'gwHealth','gwStatus',
    'HDD ERROR','HDD ERROR RESTORED','hddandDvrNvr','hddError_history','hddStatus',
    'Healthy Device(Intrusion)','Healthy Device(Time Lock)',
    'ias','iasAlarmCreatedTime','iasf','iasFault_history','iasHealth',
    'iasOff_history','iasStatus','iasSystem','imei_id',
    'Inactive Device(Intrusion)','Inactive Device(Time Lock)',
    'inactiveDeviceName','inactiveReason','inactiveSince','inactivityAlarmTime',
    # ── NEW v11: Integrated Alarm System (32% coverage) ───────────────────────
    'integratedStatus','integratedType',
    'INTEGRATED ALARM SYSTEM OFF','INTEGRATED ALARM SYSTEM ON',
    'INTEGRATED ALARM SYSTEM FAULT CONDITION RESTORED',
    'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED',
    'INTEGRATED ALARM SYSTEM ACTIVE',
    'INTRUSION ALARM FAULT CONDITION RESTORED',
    'INTRUSION ALARM SYSTEM ACTIVATE','INTRUSION ALARM SYSTEM ACTIVATION RESTORED',
    'INTRUSION ALARM SYSTEM ACTIVE','INTRUSION ALARM SYSTEM FAULT',
    'INTRUSION ALARM SYSTEM OFF','INTRUSION ALARM SYSTEM ON',
    'intrusionStatus','intrusionType',
    'lastActivityTime','lastConnectTime','lastDisconnectTime','lastUpdate',
    'lowDurationCameras','MAINS ON','major','nbgName',
    'NETWORK','notification','nvrStatus','nvrType','org_id','POWER OFF',
    'provisionState','severity','status','subsystems','SYSTEM ON','systemHealth',
    'TIME LOCK DOOR CLOSE','TIME LOCK DOOR OPEN',
    'TIME LOCK SYSTEM OFF','TIME LOCK SYSTEM ON',
    'TIME LOCK SYSTEM TAMPER','TIME LOCK TAMPER RESTORED',
    'timeLock','timeLockAlarmCreatedTime','timeLockDoor','timeLockHealth',
    'timeLockMiliTime','timeLockStatus',
    'tlsDoorOpen_history','tlsOff_history','tlsTamper_history',
    'tlStatus','tlType',
    'Total System(Intrusion)','Total System(Time Lock)',
    'ts','type','undefined','unknown','unknown_status',
    'usage_history','usage_daily','usage_last_7_days','usage_last_15_days',
    'customer_title','warning','zoName','zone_name',
]

# ══════════════════════════════════════════════════════════════════════════════
# TELEMETRY KEYS (v9 + new)
# ══════════════════════════════════════════════════════════════════════════════
TELEMETRY_KEYS = [
    'target_sw_tag','target_sw_title','target_sw_ts','target_sw_version',
    'sw_state','sw_version','fw_version','fw_state',
    'cavlidata_ontime','Total_Data_Usage',
    'sim_iccid','sim_operator','signal_strength','network_type','ip_address',
    'arrLat','arrLon','latitude','longitude',
    'cpu','ram','disk','temperature','uptime','battery_voltage',
    'memUsage','cpuUsage',
    'BAS_Downtime_Minutes','NVR_Downtime_Minutes','FAS_Downtime_Minutes',
    'IAS_Downtime_Minutes','ACS_Downtime_Minutes',
    'cameraCount','cameraOnline','cameraOffline',
    'nvrStatus','hddStatus','hddCapacity','hddUsed','recordingStatus',
    'gwStatus','powerStatus','upsStatus',
    'fasStatus','iasStatus','basStatus','accessControlStatus','timeLockStatus',
    'usage_history','lastUpdate','inactiveSince','inactiveReason',
    # ── NEW v11 telemetry ─────────────────────────────────────────────────────
    'integratedStatus','integratedType',
    'tailscale_ip','tailscale_hostname',
    'Dahua_NVR_cameraInfo',
]

# Auth
session        = requests.Session()
session.verify = False
resp = session.post(f'{TB_HOST}/api/auth/login',
    json={'username':TB_EMAIL,'password':TB_PASSWORD},
    headers={'Content-Type':'application/json'}, timeout=15)
if resp.status_code != 200:
    raise Exception(f'Login failed HTTP {resp.status_code}: {resp.text}')
JWT_TOKEN    = resp.json()['token']
AUTH_HEADERS = {'X-Authorization':f'Bearer {JWT_TOKEN}',
                'Content-Type':'application/json'}
print(f'✅ Authenticated — {TB_HOST}')
print(f'   CLIENT keys  : {len(CLIENT_KEYS)}')
print(f'   SERVER keys  : {len(SERVER_KEYS)}')
print(f'   TELEMETRY    : {len(TELEMETRY_KEYS)}')
print(f'   Bank configs : {len(BANK_HIERARCHY)}')
print(f'\n   NEW in v11:')
print(f'   • Dahua NVR client attrs   : 12 keys')
print(f'   • Tailscale VPN            : 2 keys')
print(f'   • SW OTA metadata          : 6 keys')
print(f'   • Integrated Alarm System  : 8 keys')
print(f'   • ACS tamper events        : 2 keys')
print(f'   • Mili timestamps          : 3 keys')
print(f'   • error / res fields       : 2 keys')
print(f'   • integratedStatus scoring : ✅')


✅ Authenticated — https://seple.iot-private.cloud
   CLIENT keys  : 35
   SERVER keys  : 197
   TELEMETRY    : 57
   Bank configs : 12

   NEW in v11:
   • Dahua NVR client attrs   : 12 keys
   • Tailscale VPN            : 2 keys
   • SW OTA metadata          : 6 keys
   • Integrated Alarm System  : 8 keys
   • ACS tamper events        : 2 keys
   • Mili timestamps          : 3 keys
   • error / res fields       : 2 keys
   • integratedStatus scoring : ✅


---
## Cell 4 — Core Helpers (identical to v9)

In [4]:
def safe_float(v, d=0.0):
    try:   return float(v or 0)
    except: return d

def safe_int(v, d=0):
    try:   return int(float(v or 0))
    except: return d

def to_json(v):
    if isinstance(v,(dict,list)): return v
    if isinstance(v,str):
        try: return json.loads(v)
        except: return None
    return None

def epoch_ms(ts):
    try:
        if ts and float(ts) > 0:
            return datetime.utcfromtimestamp(float(ts)/1000).strftime('%Y-%m-%d %H:%M')
    except: pass
    return ''

def is_fault(v):
    if v is None: return False
    return str(v).strip().upper() in (
        'OFFLINE','OFF','FAULT','ERROR','INACTIVE',
        'DISCONNECTED','DOWN','FAILED','0','FALSE','N/A','FAILED_UPDATE')

def is_active(v):
    return v not in (None,False,'false','False',0,'0','','null')

def first(*vals):
    for v in vals:
        if v not in (None,'','null','None'): return v
    return ''

def paginate(url_tpl, page_size=100):
    items, page = [], 0
    while True:
        url  = url_tpl.format(page=page, size=page_size)
        resp = session.get(url, headers=AUTH_HEADERS, timeout=30)
        if resp.status_code == 401: raise Exception('JWT expired — re-run Cell 3')
        if resp.status_code != 200:
            print(f'  ⚠️ HTTP {resp.status_code}: {url[:80]}')
            break
        data = resp.json()
        items.extend(data.get('data',[]))
        if not data.get('hasNext',False): break
        page += 1
        time.sleep(REQUEST_DELAY)
    return items

def fetch_attr_scope_keys(etype, eid, scope, keys):
    result = {}
    for i in range(0, len(keys), 50):
        chunk = keys[i:i+50]
        url   = (f'{TB_HOST}/api/plugins/telemetry/{etype}/{eid}'
                 f'/values/attributes/{scope}?keys={",".join(chunk)}')
        try:
            r = session.get(url, headers=AUTH_HEADERS, timeout=20)
            if r.status_code == 200:
                for item in r.json():
                    result[item['key']] = item['value']
        except: pass
        time.sleep(0.02)
    return result

def fetch_attr_scope_all(etype, eid, scope):
    url = f'{TB_HOST}/api/plugins/telemetry/{etype}/{eid}/values/attributes/{scope}'
    try:
        r = session.get(url, headers=AUTH_HEADERS, timeout=15)
        if r.status_code == 200:
            return {item['key']: item['value'] for item in r.json()}
    except: pass
    return {}

def fetch_all_attributes(etype, eid):
    attrs = {}
    attrs.update(fetch_attr_scope_keys(etype, eid, 'CLIENT_SCOPE', CLIENT_KEYS))
    attrs.update(fetch_attr_scope_keys(etype, eid, 'SERVER_SCOPE', SERVER_KEYS))
    for scope in ['SERVER_SCOPE','CLIENT_SCOPE','SHARED_SCOPE']:
        for k, v in fetch_attr_scope_all(etype, eid, scope).items():
            if k not in attrs:
                attrs[k] = v
    return attrs

def fetch_telemetry(device_id, keys):
    result = {}
    for i in range(0, len(keys), 50):
        chunk = keys[i:i+50]
        url   = (f'{TB_HOST}/api/plugins/telemetry/DEVICE/{device_id}'
                 f'/values/timeseries?keys={",".join(chunk)}&useStrictDataTypes=false')
        try:
            r = session.get(url, headers=AUTH_HEADERS, timeout=20)
            if r.status_code == 200:
                for k, entries in r.json().items():
                    if entries:
                        result[f'tele_{k}']    = entries[0].get('value')
                        result[f'tele_{k}_ts'] = epoch_ms(entries[0].get('ts'))
        except: pass
        time.sleep(0.02)
    return result

_rel_cache = {}
def get_parents(etype, eid):
    key = (etype, eid)
    if key in _rel_cache: return _rel_cache[key]
    url = f'{TB_HOST}/api/relations?toId={eid}&toType={etype}'
    try:
        r = session.get(url, headers=AUTH_HEADERS, timeout=15)
        if r.status_code == 200:
            parents = [{'entity_type': rel.get('from',{}).get('entityType',''),
                        'entity_id':   rel.get('from',{}).get('id','')}
                       for rel in (r.json() if isinstance(r.json(), list) else [])]
            _rel_cache[key] = parents
            time.sleep(REQUEST_DELAY)
            return parents
    except: pass
    _rel_cache[key] = []
    return []

print('✅ Core helpers ready.')


✅ Core helpers ready.


---
## Cell 5 — Fetch Customers (Banks)

In [5]:
from tqdm import tqdm

print('📡 Fetching Customers (Banks) ...')
raw_custs = paginate(
    TB_HOST + '/api/customers?pageSize={size}&page={page}&sortProperty=title&sortOrder=ASC'
)
print(f'   Found {len(raw_custs)} customers.\n')

customers = []
for c in tqdm(raw_custs, desc='Customers'):
    cid   = c.get('id',{}).get('id','')
    title = c.get('title','')
    attrs = fetch_attr_scope_all('CUSTOMER', cid, 'SERVER_SCOPE')
    attrs.update(fetch_attr_scope_all('CUSTOMER', cid, 'CLIENT_SCOPE'))

    bank_cfg = None
    for bank_name, cfg in BANK_HIERARCHY.items():
        aliases = cfg.get('aliases', [])
        tlow = title.lower()
        if (bank_name.lower() in tlow or tlow in bank_name.lower()
                or any(a.lower() in tlow for a in aliases)):
            bank_cfg = cfg
            bank_cfg['bank_name'] = bank_name
            break

    customers.append({
        'customer_id':    cid,
        'customer_title': title,
        'bank_name':      bank_cfg['bank_name'] if bank_cfg else title,
        'bank_depth':     bank_cfg['depth']     if bank_cfg else 3,
        'nbg_name':       first(attrs.get('nbgName'), attrs.get('nbg_name'), title),
        'region':         first(attrs.get('region'),  attrs.get('circle'),   ''),
        'state':          first(attrs.get('state'),   c.get('state',''),     ''),
        'city':           first(attrs.get('city'),    c.get('city',''),      ''),
        'address':        first(attrs.get('address'), c.get('address',''),   ''),
        'email':          first(attrs.get('email'),   c.get('email',''),     ''),
        'phone':          first(attrs.get('phone'),   c.get('phone',''),     ''),
        'created_time':   epoch_ms(c.get('createdTime')),
        '_attrs':         attrs,
    })
    time.sleep(REQUEST_DELAY)

cust_map   = {c['customer_id']: c for c in customers}
cust_title = {c['customer_id']: c['customer_title'] for c in customers}
cust_bank  = {c['customer_id']: c['bank_name'] for c in customers}
print(f'\n✅ {len(customers)} customers fetched.')


📡 Fetching Customers (Banks) ...


   Found 189 customers.



Customers:   0%|                                                                               | 0/189 [00:00<?, ?it/s]

Customers:   1%|▍                                                                      | 1/189 [00:00<00:28,  6.61it/s]

Customers:   1%|▊                                                                      | 2/189 [00:00<00:31,  6.02it/s]

...

Customers:  99%|████████████████████████████████████████████████████████████████████▎| 187/189 [00:30<00:00,  6.78it/s]

Customers:  99%|████████████████████████████████████████████████████████████████████▋| 188/189 [00:31<00:00,  6.86it/s]

Customers: 100%|█████████████████████████████████████████████████████████████████████| 189/189 [00:31<00:00,  6.70it/s]

Customers: 100%|█████████████████████████████████████████████████████████████████████| 189/189 [00:31<00:00,  6.05it/s]


✅ 189 customers fetched.


---
## Cell 6 — Fetch Assets + Classify

In [6]:
print('📡 Fetching ALL Assets ...')
raw_assets = paginate(
    TB_HOST + '/api/tenant/assets?pageSize={size}&page={page}&sortProperty=name&sortOrder=ASC'
)
print(f'   Found {len(raw_assets)} assets. Classifying ...\n')

assets = []
for a in tqdm(raw_assets, desc='Assets'):
    aid   = a.get('id',{}).get('id','')
    atype = a.get('type','')
    cid   = (a.get('customerId') or {}).get('id','')
    bank  = cust_bank.get(cid, '')
    level = get_asset_level(atype, bank)
    if level == '_ignore': continue

    attrs = {}
    for scope in ['SERVER_SCOPE','CLIENT_SCOPE']:
        attrs.update(fetch_attr_scope_all('ASSET', aid, scope))

    assets.append({
        'asset_id':    aid,  'asset_name':  a.get('name',''),
        'asset_type':  atype,'level':       level,
        'customer_id': cid,  'bank_name':   bank,
        'created':     epoch_ms(a.get('createdTime')),
        'nbg_name':    first(attrs.get('nbgName'),    attrs.get('nbg_name'),   ''),
        'zo_name':     first(attrs.get('zoName'),     attrs.get('zo_name'),
                             attrs.get('zoneName'),   ''),
        'zo_code':     first(attrs.get('zoCode'),     attrs.get('zone_code'),  ''),
        'ro_name':     first(attrs.get('roName'),     attrs.get('ro_name'),    ''),
        'branch_name': first(attrs.get('branchName'), attrs.get('branch_name'),''),
        'branch_code': first(attrs.get('branchCode'), attrs.get('branch_code'),''),
        'display_name':first(attrs.get('displayName'), a.get('name',''),       ''),
        'address':     first(attrs.get('address'),    ''),
        'city':        first(attrs.get('city'),       ''),
        'state':       first(attrs.get('state'),      ''),
        'pincode':     first(attrs.get('pincode'),    attrs.get('zip',''),     ''),
        'latitude':    first(attrs.get('latitude'),   attrs.get('arrLat'),     ''),
        'longitude':   first(attrs.get('longitude'),  attrs.get('arrLon'),     ''),
        'install_date':first(attrs.get('installationDate'), ''),
        'go_live_date':first(attrs.get('goLiveDate'),       ''),
        'contract':    first(attrs.get('contractType'),     ''),
        'sla':         first(attrs.get('slaTier'),          ''),
        '_attrs':      attrs,
    })
    time.sleep(REQUEST_DELAY)

asset_map  = {a['asset_id']: a for a in assets}
level_dist = Counter(a['level'] for a in assets)
print(f'\n✅ {len(assets)} assets classified. Levels:')
for lvl, cnt in sorted(level_dist.items()):
    print(f'   {lvl:<15} {cnt}')


📡 Fetching ALL Assets ...


   Found 160 assets. Classifying ...



Assets:   0%|                                                                                  | 0/160 [00:00<?, ?it/s]

Assets:   1%|▍                                                                         | 1/160 [00:00<00:25,  6.35it/s]

Assets:   1%|▉                                                                         | 2/160 [00:00<00:26,  5.92it/s]

...

Assets:  99%|███████████████████████████████████████████████████████████████████████ | 158/160 [00:26<00:00,  6.59it/s]

Assets:  99%|███████████████████████████████████████████████████████████████████████▌| 159/160 [00:26<00:00,  5.99it/s]

Assets: 100%|████████████████████████████████████████████████████████████████████████| 160/160 [00:26<00:00,  6.23it/s]

Assets: 100%|████████████████████████████████████████████████████████████████████████| 160/160 [00:26<00:00,  5.97it/s]


✅ 157 assets classified. Levels:
   branch          120
   ho              5
   nbg             6
   ro              8
   zo              18


---
## Cell 7 — Fetch All Devices + Attributes + Telemetry

In [7]:
print('📡 Fetching ALL Devices ...')
all_devices = paginate(
    TB_HOST + '/api/tenant/devices?pageSize={size}&page={page}&sortProperty=name&sortOrder=ASC'
)
print(f'   Found {len(all_devices)} devices. Fetching all data ...\n')

device_data, fetch_errors = [], []
for device in tqdm(all_devices, desc='Devices', unit='dev'):
    dev_id   = device.get('id',{}).get('id','')
    dev_name = device.get('name','UNKNOWN')
    cust_id  = (device.get('customerId') or {}).get('id','')
    bank     = cust_bank.get(cust_id, '')
    try:
        attrs  = fetch_all_attributes('DEVICE', dev_id)
        tele   = fetch_telemetry(dev_id, TELEMETRY_KEYS)
        merged = {**tele, **attrs}  # attrs win
        merged.update({
            '_device_id':      dev_id,
            '_device_name':    dev_name,
            '_device_type':    device.get('type',''),
            '_device_profile': device.get('deviceProfileName',''),
            '_customer_id':    cust_id,
            '_customer_name':  cust_title.get(cust_id,''),
            '_bank_name':      bank,
            '_created_time':   device.get('createdTime',''),
        })
        device_data.append(merged)
    except Exception as e:
        fetch_errors.append({'id':dev_id,'name':dev_name,'error':str(e)})
    time.sleep(REQUEST_DELAY)

dev_index = {d['_device_id']: d for d in device_data}
print(f'\n✅ Fetched : {len(device_data)} devices | Errors: {len(fetch_errors)}')
all_keys = set(k for d in device_data for k in d)
print(f'   Unique keys across all devices: {len(all_keys)}')


📡 Fetching ALL Devices ...


   Found 184 devices. Fetching all data ...



Devices:   0%|                                                                                | 0/184 [00:00<?, ?dev/s]

Devices:   1%|▍                                                                       | 1/184 [00:00<02:07,  1.44dev/s]

Devices:   1%|▊                                                                       | 2/184 [00:01<02:11,  1.38dev/s]

...

Devices:  99%|█████████████████████████████████████████████████████████████████████▏| 182/184 [02:32<00:01,  1.41dev/s]

Devices:  99%|█████████████████████████████████████████████████████████████████████▌| 183/184 [02:33<00:00,  1.44dev/s]

Devices: 100%|██████████████████████████████████████████████████████████████████████| 184/184 [02:33<00:00,  1.43dev/s]

Devices: 100%|██████████████████████████████████████████████████████████████████████| 184/184 [02:33<00:00,  1.19dev/s]


✅ Fetched : 184 devices | Errors: 0
   Unique keys across all devices: 610


---
## Cell 8 — Hierarchy Resolver (bank-aware, identical to v9)

In [8]:
def resolve_hierarchy_walk(etype, eid, bank_name='', depth=0):
    if depth >= MAX_RELATION_DEPTH: return {}
    result = {}
    for p in get_parents(etype, eid):
        ptype, pid = p['entity_type'], p['entity_id']
        if ptype == 'CUSTOMER' and pid in cust_map:
            c = cust_map[pid]
            result.setdefault('bank_name', c['bank_name'])
            result.setdefault('nbg_name',  c['nbg_name'])
            clvl  = classify_entity_level(c['customer_title'])
            cname = c['customer_title']
            if   clvl == 'lho': result.setdefault('lho_name', cname)
            elif clvl == 'rbo': result.setdefault('rbo_name', cname)
            elif clvl == 'zo':  result.setdefault('zo_name',  cname)
            elif clvl == 'ro':  result.setdefault('ro_name',  cname)
            elif clvl == 'co':  result.setdefault('co_name',  cname)
            elif clvl == 'ho':  result.setdefault('ho_name',  cname)
            elif clvl == 'nbg': result.setdefault('nbg_name', cname)
        elif ptype == 'ASSET' and pid in asset_map:
            a, level = asset_map[pid], asset_map[pid]['level']
            if level == '_ignore': continue
            name = a['display_name'] or a['asset_name']
            if   level == 'ho':  result.setdefault('ho_name',   name); result.setdefault('ho_id',  pid)
            elif level == 'nbg': result.setdefault('nbg_name',  first(a['nbg_name'],name)); result.setdefault('nbg_id',  pid)
            elif level == 'lho': result.setdefault('lho_name',  name); result.setdefault('lho_id', pid)
            elif level == 'zo':  result.setdefault('zo_name',   first(a['zo_name'],name));  result.setdefault('zo_code', a['zo_code']); result.setdefault('zo_state',a['state'])
            elif level == 'ro':  result.setdefault('ro_name',   first(a['ro_name'],name));  result.setdefault('ro_id',  pid)
            elif level == 'co':  result.setdefault('co_name',   name); result.setdefault('co_id',  pid)
            elif level == 'rbo': result.setdefault('rbo_name',  name); result.setdefault('rbo_id', pid)
            elif level in ('branch','sub_branch'):
                result.setdefault('branch_name',    first(a['branch_name'],name))
                result.setdefault('branch_code',    a['branch_code'])
                result.setdefault('branch_address', a['address'])
                result.setdefault('branch_city',    a['city'])
                result.setdefault('branch_state',   a['state'])
                result.setdefault('branch_pincode', a['pincode'])
                result.setdefault('branch_lat',     a['latitude'])
                result.setdefault('branch_lon',     a['longitude'])
                result.setdefault('install_date',   a['install_date'])
                result.setdefault('go_live_date',   a['go_live_date'])
                result.setdefault('contract',       a['contract'])
                result.setdefault('sla',            a['sla'])
            upper = resolve_hierarchy_walk('ASSET', pid, bank_name, depth+1)
            for k, v in upper.items(): result.setdefault(k, v)
    return result

def build_hierarchy(attrs):
    h = {
        'bank_name':     attrs.get('_bank_name',''),
        'nbg_name':      first(attrs.get('nbgName'),    attrs.get('nbg_name'),   ''),
        'zo_name':       first(attrs.get('zoName'),     attrs.get('zo_name'),    attrs.get('zoneName'),''),
        'zo_code':       first(attrs.get('zoCode'),     attrs.get('zone_code'),  ''),
        'branch_name':   first(attrs.get('branchName'), attrs.get('branch_name'),attrs.get('formattedBranchName'),''),
        'branch_code':   first(attrs.get('branch_id'),  attrs.get('branchCode'), ''),
        'branch_address':first(attrs.get('address'),    attrs.get('addr'),        ''),
        'branch_city':   first(attrs.get('city'),       ''),
        'branch_state':  first(attrs.get('state'),      ''),
        'branch_lat':    first(attrs.get('arrLat'),     attrs.get('latitude'),    ''),
        'branch_lon':    first(attrs.get('arrLon'),     attrs.get('longitude'),   ''),
        'ho_name':'','lho_name':'','ro_name':'','co_name':'','rbo_name':'',
    }
    if not h['nbg_name'] or not h['branch_name']:
        rel = resolve_hierarchy_walk('DEVICE', attrs['_device_id'], attrs.get('_bank_name',''))
        for k, v in rel.items():
            if not h.get(k) and v: h[k] = v
    cid = attrs.get('_customer_id','')
    if not h['nbg_name'] and cid in cust_map:
        c = cust_map[cid]
        h['nbg_name'] = c['nbg_name']
        h['bank_name'] = c['bank_name']
    path_parts = [h.get(f) for f in ['bank_name','ho_name','nbg_name','lho_name',
                                      'zo_name','ro_name','co_name','rbo_name','branch_name']
                  if h.get(f)]
    h['full_path']       = ' → '.join(path_parts)
    h['hierarchy_depth'] = len(path_parts)
    return h

print('🔗 Resolving hierarchy ...')
for d in tqdm(device_data, desc='Hierarchy', unit='dev'):
    d['_hierarchy'] = build_hierarchy(d)

total = len(device_data)
def pct(n): return f'{n}/{total} ({100*n//max(total,1)}%)'
print(f'\n✅ Hierarchy resolved:')
print(f'   Bank/NBG : {pct(sum(1 for d in device_data if d["_hierarchy"].get("nbg_name")))}')
print(f'   Zone     : {pct(sum(1 for d in device_data if d["_hierarchy"].get("zo_name")))}')
print(f'   Branch   : {pct(sum(1 for d in device_data if d["_hierarchy"].get("branch_name")))}')
print('\nSample full paths:')
for d in device_data[:6]:
    print(f'  {d["_hierarchy"]["full_path"] or "(unresolved)"}')


🔗 Resolving hierarchy ...


Hierarchy:   0%|                                                                              | 0/184 [00:00<?, ?dev/s]

Hierarchy:  13%|████████▊                                                           | 24/184 [00:00<00:01, 131.08dev/s]

Hierarchy:  21%|██████████████                                                      | 38/184 [00:00<00:01, 111.27dev/s]

Hierarchy:  52%|███████████████████████████████████                                 | 95/184 [00:00<00:00, 266.61dev/s]

...

Hierarchy:  93%|███████████████████████████████████████████████████████████████▌    | 172/184 [00:05<00:00, 19.50dev/s]

Hierarchy:  97%|██████████████████████████████████████████████████████████████████▏ | 179/184 [00:05<00:00, 17.14dev/s]

Hierarchy: 100%|████████████████████████████████████████████████████████████████████| 184/184 [00:06<00:00, 15.79dev/s]

Hierarchy: 100%|████████████████████████████████████████████████████████████████████| 184/184 [00:06<00:00, 28.61dev/s]


✅ Hierarchy resolved:
   Bank/NBG : 134/184 (72%)
   Zone     : 129/184 (70%)
   Branch   : 121/184 (65%)

Sample full paths:
  BANK OF BARODA → NBG EAST → ZO HOWRAH → ABC
  NBG EAST → ZO HOWRAH → Branch TR
  TLS CUS DEMO → TLS CUS DEMO → ZO Muzaffapur
  BANK OF BARODA → ZO(Kolkata) → ZO KOLKATA → BRANCH_NSB AIRPORT
  BANK OF BARODA → Bank of Baroda demo → ZO KOLKATA → BRANCH AMTALA
  BANK OF BARODA → ZO(Kolkata) → RO(KMR) → BRANCH_APC ROAD


---
## Cell 9 — Build Master DataFrame
### NEW v11 columns: Dahua NVR, integratedStatus, tailscale, ACS tamper, mili times, sw metadata

In [9]:
import pandas as pd

EVENT_FLAG_MAP = {
    'ev_power_off':           'POWER OFF',
    'ev_dvr_off':             'DVR/NVR OFF',
    'ev_dvr_on':              'DVR/NVR ON',
    'ev_hdd_error':           'HDD ERROR',
    'ev_hdd_restored':        'HDD ERROR RESTORED',
    'ev_battery_low':         'BATTERY LOW',
    'ev_battery_reverse':     'BATTERY REVERSE',
    'ev_battery_on':          'BATTERY ON',
    'ev_mains_on':            'MAINS ON',
    'ev_system_on':           'SYSTEM ON',
    'ev_network':             'NETWORK',
    'ev_cam_disconnect':      'CAMERA DISCONNECT',
    'ev_cam_tamper':          'CAMERA TAMPER',
    'ev_cam_tamper_rst':      'CAMERA TAMPERED RESTORED',
    'ev_cam_connect':         'CAMERA CONNECTION ESTABLISHED',
    'ev_fas_off':             'FIRE ALARM SYSTEM OFF',
    'ev_fas_on':              'FIRE ALARM SYSTEM ON',
    'ev_fas_fault':           'FIRE ALARM SYSTEM FAULT',
    'ev_fas_fault_rst':       'FIRE ALARM SYSTEM FAULT CONDITION RESTORED',
    'ev_fas_active':          'FIRE ALARM SYSTEM ACTIVE',
    'ev_fas_activate':        'FIRE ALARM SYSTEM ACTIVATE',
    'ev_fas_activate_rst':    'FIRE ALARM SYSTEM ACTIVATION RESTORED',
    'ev_ias_off':             'INTRUSION ALARM SYSTEM OFF',
    'ev_ias_on':              'INTRUSION ALARM SYSTEM ON',
    'ev_ias_fault':           'INTRUSION ALARM SYSTEM FAULT',
    'ev_ias_fault_rst':       'INTRUSION ALARM FAULT CONDITION RESTORED',
    'ev_ias_active':          'INTRUSION ALARM SYSTEM ACTIVE',
    'ev_ias_activate':        'INTRUSION ALARM SYSTEM ACTIVATE',
    'ev_ias_activate_rst':    'INTRUSION ALARM SYSTEM ACTIVATION RESTORED',
    # Integrated Alarm System events
    'ev_int_off':             'INTEGRATED ALARM SYSTEM OFF',
    'ev_int_on':              'INTEGRATED ALARM SYSTEM ON',
    'ev_int_fault_rst':       'INTEGRATED ALARM SYSTEM FAULT CONDITION RESTORED',
    'ev_int_act_rst':         'INTEGRATED ALARM SYSTEM ACTIVATION RESTORED',
    'ev_int_active':          'INTEGRATED ALARM SYSTEM ACTIVE',
    # Time Lock
    'ev_tls_off':             'TIME LOCK SYSTEM OFF',
    'ev_tls_on':              'TIME LOCK SYSTEM ON',
    'ev_tls_tamper':          'TIME LOCK SYSTEM TAMPER',
    'ev_tls_tamper_rst':      'TIME LOCK TAMPER RESTORED',
    'ev_tls_door_open':       'TIME LOCK DOOR OPEN',
    'ev_tls_door_close':      'TIME LOCK DOOR CLOSE',
    # ── NEW v11: ACS tamper ────────────────────────────────────────────────────
    'ev_acs_tamper_rst':      'ACCESS CONTROL SYSTEM TAMPER RESTORED',
}

print(f'⚙️  Building master DataFrame for {len(device_data)} devices ...')
rows = []
for attrs in device_data:
    h  = attrs['_hierarchy']
    sh = to_json(attrs.get('systemHealth')) or {}

    def tele(k, default=None):
        return attrs.get(f'tele_{k}', attrs.get(k, default))

    ch_dc = {f'camDC_ch{i}': json.dumps(to_json(attrs.get(f'cameraDisconnectCH{i}_history')) or [])
             for i in range(1,17)}
    ch_tp = {f'camTP_ch{i}': json.dumps(to_json(attrs.get(f'cameraTamperCH{i}_history')) or [])
             for i in range(1,17)}

    row = {
        # ── HIERARCHY ────────────────────────────────────────────────────────
        'bank_name':      h.get('bank_name',  attrs.get('_bank_name','')),
        'ho_name':        h.get('ho_name',    ''),
        'nbg_name':       h.get('nbg_name',   ''),
        'lho_name':       h.get('lho_name',   ''),
        'zo_name':        h.get('zo_name',    ''),
        'zo_code':        h.get('zo_code',    ''),
        'zo_state':       h.get('zo_state',   ''),
        'ro_name':        h.get('ro_name',    ''),
        'co_name':        h.get('co_name',    ''),
        'rbo_name':       h.get('rbo_name',   ''),
        'branch_name':    h.get('branch_name', attrs.get('_device_name','')),
        'branch_code':    h.get('branch_code',''),
        'branch_address': h.get('branch_address',''),
        'branch_city':    h.get('branch_city',''),
        'branch_state':   h.get('branch_state',''),
        'branch_pincode': h.get('branch_pincode',''),
        'branch_lat':     h.get('branch_lat', ''),
        'branch_lon':     h.get('branch_lon', ''),
        'install_date':   h.get('install_date',''),
        'go_live_date':   h.get('go_live_date',''),
        'contract_type':  h.get('contract',   ''),
        'sla_tier':       h.get('sla',        ''),
        'full_path':      h.get('full_path',  ''),
        'hierarchy_depth':h.get('hierarchy_depth',0),

        # ── DEVICE IDENTITY ──────────────────────────────────────────────────
        'device_id':      attrs['_device_id'],
        'device_name':    attrs['_device_name'],
        'device_type':    attrs.get('_device_type',''),
        'device_profile': attrs.get('_device_profile',''),
        'customer_id':    attrs['_customer_id'],
        'customer_name':  attrs.get('_customer_name',''),
        'device_created': epoch_ms(attrs.get('_created_time')),
        'org_id':         first(attrs.get('org_id'),''),
        'imei_id':        first(attrs.get('imei_id'),''),
        'provisionState': first(attrs.get('provisionState'),''),
        'active':         first(attrs.get('active'),''),
        'device_status':  first(attrs.get('status'),''),
        # ── NEW v11: error / res fields ────────────────────────────────────────
        'device_error':   first(attrs.get('error'),''),
        'device_res':     first(attrs.get('res'),''),

        # ── HIKVISION NVR (client attrs) ─────────────────────────────────────
        'hik_model':       first(attrs.get('Hikvision_NVR_model'),     attrs.get('nvrType'),''),
        'hik_serial':      first(attrs.get('Hikvision_NVR_serialNumber'),''),
        'hik_firmware':    first(attrs.get('Hikvision_NVR_firmwareVersion'),''),
        'hik_hardware':    first(attrs.get('Hikvision_NVR_hardwareVersion'),''),
        'hik_mac':         first(attrs.get('Hikvision_NVR_macAddress'),''),
        'hik_manufacturer':first(attrs.get('Hikvision_NVR_Manufacturer'),''),
        'hik_processor':   first(attrs.get('Hikvision_NVR_Processor'),''),
        'hik_device_id':   first(attrs.get('Hikvision_NVR_deviceID'),''),
        'hik_hdd_info':    json.dumps(to_json(attrs.get('Hikvision_NVR_HDDInfo')) or ''),
        'hik_cam_info':    json.dumps(to_json(attrs.get('Hikvision_NVR_cameraInfo')) or []),

        # ── NEW v11: DAHUA NVR (client attrs, 36% coverage) ───────────────────
        'dahua_model':      first(attrs.get('Dahua_NVR_model'),''),
        'dahua_serial':     first(attrs.get('Dahua_NVR_serialNumber'),''),
        'dahua_firmware':   first(attrs.get('Dahua_NVR_firmwareVersion'),''),
        'dahua_hardware':   first(attrs.get('Dahua_NVR_hardwareVersion'),''),
        'dahua_mac':        first(attrs.get('Dahua_NVR_macAddress'),''),
        'dahua_manufacturer':first(attrs.get('Dahua_NVR_Manufacturer'),''),
        'dahua_processor':  first(attrs.get('Dahua_NVR_Processor'),''),
        'dahua_device_id':  first(attrs.get('Dahua_NVR_deviceID'),''),
        'dahua_hdd_info':   json.dumps(to_json(attrs.get('Dahua_NVR_HDDInfo')) or ''),
        'dahua_cam_info':   json.dumps(to_json(attrs.get('Dahua_NVR_cameraInfo',
                                      tele('Dahua_NVR_cameraInfo'))) or []),
        # NVR brand detection
        'nvr_brand':        ('Dahua' if attrs.get('Dahua_NVR_model')
                             else 'Hikvision' if attrs.get('Hikvision_NVR_model')
                             else ''),

        # ── NEW v11: TAILSCALE VPN (35% coverage) ─────────────────────────────
        'tailscale_hostname':first(attrs.get('tailscale_hostname'), tele('tailscale_hostname'),''),
        'tailscale_ip':      first(attrs.get('tailscale_ip'),       tele('tailscale_ip'),''),

        # ── NEW v11: SW OTA METADATA (35% coverage) ───────────────────────────
        'sw_id':             first(attrs.get('sw_id'),''),
        'sw_checksum':       first(attrs.get('sw_checksum'),''),
        'sw_checksum_algo':  first(attrs.get('sw_checksum_algorithm'),''),
        'sw_size':           safe_float(attrs.get('sw_size',0)),

        # ── SUBSYSTEM STATUSES ───────────────────────────────────────────────
        'nvr_status':        first(attrs.get('nvrStatus'),     tele('nvrStatus'),''),
        'hdd_status':        first(attrs.get('hddStatus'),     tele('hddStatus'),''),
        'hdd_capacity':      safe_float(tele('hddCapacity',0)),
        'hdd_used':          safe_float(tele('hddUsed',0)),
        'fas_status':        first(attrs.get('fasStatus'),     tele('fasStatus'),''),
        'fas_health':        first(attrs.get('fasHealth'),''),
        'fas_system':        first(attrs.get('fasSystem'),''),
        'fire_alarm_status': first(attrs.get('fireAlarmStatus'),''),
        'fire_alarm_type':   first(attrs.get('fireAlarmType'),''),
        'ias_status':        first(attrs.get('iasStatus'),     tele('iasStatus'),''),
        'ias_health':        first(attrs.get('iasHealth'),''),
        'ias_system':        first(attrs.get('iasSystem'),''),
        'intrusion_status':  first(attrs.get('intrusionStatus'),''),
        'intrusion_type':    first(attrs.get('intrusionType'),''),
        'bas_status':        first(attrs.get('basStatus'),     tele('basStatus'),''),
        'bas_health':        first(attrs.get('basHealth'),''),
        'bas_system':        first(attrs.get('basSystem'),''),
        'acs_status':        first(attrs.get('accessControlStatus'),tele('accessControlStatus'),''),
        'acs_health':        first(attrs.get('accessControlHealth'),''),
        'acs_door':          first(attrs.get('accessControlDoor'),''),
        # ── NEW v11: ACS created time ──────────────────────────────────────────
        'acs_created_time':  epoch_ms(attrs.get('accessControlCreatedTime')),
        'tls_status':        first(attrs.get('timeLockStatus'), attrs.get('tlStatus'),
                                   tele('timeLockStatus'),''),
        'tls_health':        first(attrs.get('timeLockHealth'),''),
        'tls_door':          first(attrs.get('timeLockDoor'),''),
        'tls_type':          first(attrs.get('tlType'),''),
        'gw_status':         first(attrs.get('gwStatus'),       attrs.get('gatewayStatus'),
                                   tele('gwStatus'),''),
        'gw_health':         first(attrs.get('gwHealth'),''),
        'gw_type':           first(attrs.get('gatewayType'),''),
        'cctv_status':       first(attrs.get('cctvStatus'),     tele('cctvStatus'),''),
        'power_status':      first(attrs.get('powerStatus'),    tele('powerStatus'),''),
        'ups_status':        first(tele('upsStatus'),''),
        'recording_status':  first(tele('recordingStatus'),''),
        # ── NEW v11: Integrated Alarm System (32% coverage) ───────────────────
        'integrated_status': first(attrs.get('integratedStatus'), tele('integratedStatus'),''),
        'integrated_type':   first(attrs.get('integratedType'),   tele('integratedType'),''),

        # ── CAMERA ───────────────────────────────────────────────────────────
        'cam_total':         safe_int(first(tele('cameraCount'),  attrs.get('cameraCount'),0)),
        'cam_online':        safe_int(first(tele('cameraOnline'), attrs.get('cameraOnline'),0)),
        'cam_offline':       safe_int(first(tele('cameraOffline'),attrs.get('cameraOffline'),0)),
        'cam_dc_count':      safe_int(attrs.get('cameraDisconnectCount',0)),
        'cam_tamper_count':  safe_int(attrs.get('cameraTamperCount',0)),
        'count_ch':          safe_int(attrs.get('count_CH',0)),
        'count_hdd':         safe_int(attrs.get('count_HDD',0)),
        'cam_link_status':   json.dumps(to_json(attrs.get('cameraLinkStatus')) or {}),
        'low_dur_cameras':   str(attrs.get('lowDurationCameras','') or ''),

        # ── SYSTEM HEALTH ────────────────────────────────────────────────────
        'disk_pct':          safe_float(sh.get('disk',  tele('disk',0))),
        'cpu_pct':           safe_float(sh.get('cpu',   tele('cpu', 0))),
        'ram_pct':           safe_float(sh.get('ram',   tele('ram', 0))),
        'battery_voltage':   safe_float(sh.get('battery_voltage', tele('battery_voltage',0))),
        'temperature':       safe_float(tele('temperature',0)),
        'uptime_sec':        safe_float(tele('uptime',0)),

        # ── GPS ──────────────────────────────────────────────────────────────
        'latitude':          safe_float(first(tele('arrLat'),  tele('latitude'),  0)),
        'longitude':         safe_float(first(tele('arrLon'),  tele('longitude'), 0)),

        # ── TELEMETRY / SIM ──────────────────────────────────────────────────
        'total_data_mb':     safe_float(tele('Total_Data_Usage',0)),
        'bas_downtime_min':  safe_float(tele('BAS_Downtime_Minutes',0)),
        'nvr_downtime_min':  safe_float(tele('NVR_Downtime_Minutes',0)),
        'fas_downtime_min':  safe_float(tele('FAS_Downtime_Minutes',0)),
        'ias_downtime_min':  safe_float(tele('IAS_Downtime_Minutes',0)),
        'acs_downtime_min':  safe_float(tele('ACS_Downtime_Minutes',0)),
        'cavli_ontime':      safe_float(tele('cavlidata_ontime',0)),
        'sim_iccid':         first(tele('sim_iccid'),''),
        'sim_operator':      first(tele('sim_operator'),''),
        'signal_strength':   safe_float(tele('signal_strength',0)),
        'network_type':      first(tele('network_type'),''),
        'ip_address':        first(tele('ip_address'),''),

        # ── SOFTWARE / OTA ────────────────────────────────────────────────────
        'sw_state':          first(tele('sw_state'),''),
        'sw_version':        first(tele('sw_version'), tele('target_sw_version'),''),
        'sw_title':          first(tele('target_sw_title'), attrs.get('sw_title'),''),
        'sw_tag':            first(tele('target_sw_tag'),   attrs.get('sw_tag'),''),
        'fw_version':        first(tele('fw_version'),''),
        'fw_state':          first(tele('fw_state'),''),

        # ── TIMESTAMPS ────────────────────────────────────────────────────────
        'last_update':       first(attrs.get('lastUpdate'),     tele('lastUpdate'),''),
        'last_connect':      epoch_ms(attrs.get('lastConnectTime')),
        'last_disconnect':   epoch_ms(attrs.get('lastDisconnectTime')),
        'last_activity':     epoch_ms(attrs.get('lastActivityTime')),
        'inactive_since':    first(attrs.get('inactiveSince'),  tele('inactiveSince'),''),
        'inactive_reason':   first(attrs.get('inactiveReason'), tele('inactiveReason'),''),
        'inactivity_alarm':  epoch_ms(attrs.get('inactivityAlarmTime')),
        'bas_alarm_ts':      epoch_ms(attrs.get('basAlarmCreatedTime')),
        'fas_alarm_ts':      epoch_ms(attrs.get('fasAlarmCreatedTime')),
        'ias_alarm_ts':      epoch_ms(attrs.get('iasAlarmCreatedTime')),
        'gw_alarm_ts':       epoch_ms(attrs.get('gatewayAlarmCreatedTime')),
        'tls_alarm_ts':      epoch_ms(attrs.get('timeLockAlarmCreatedTime')),
        'cctv_alarm_ts':     epoch_ms(attrs.get('cctvAlarmCreatedTime')),
        # ── NEW v11: mili timestamps ───────────────────────────────────────────
        'gate_mili_time':    str(attrs.get('gateMiliTime','') or ''),
        'cctv_mili_time':    str(attrs.get('cctvMiliTime','') or ''),
        'tls_mili_time':     str(attrs.get('timeLockMiliTime','') or ''),

        # ── ALARM METADATA ────────────────────────────────────────────────────
        'alarm_flag':        attrs.get('alarmFlag'),
        'severity_attr':     first(attrs.get('severity'),''),
        'critical_flag':     attrs.get('critical'),
        'major_flag':        attrs.get('major'),
        'warning_flag':      attrs.get('warning'),
        'notification':      attrs.get('notification'),
        'care':              attrs.get('care'),

        # ── SUBSYSTEM COUNTS ──────────────────────────────────────────────────
        'total_sys_intrusion': attrs.get('Total System(Intrusion)',''),
        'total_sys_timelock':  attrs.get('Total System(Time Lock)',''),
        'faulty_intrusion':    attrs.get('Faulty Device(Intrusion)',''),
        'faulty_timelock':     attrs.get('Faulty Device(Time Lock)',''),
        'healthy_intrusion':   attrs.get('Healthy Device(Intrusion)',''),
        'healthy_timelock':    attrs.get('Healthy Device(Time Lock)',''),
        'inactive_intrusion':  attrs.get('Inactive Device(Intrusion)',''),
        'inactive_timelock':   attrs.get('Inactive Device(Time Lock)',''),
        'inactive_device_name':attrs.get('inactiveDeviceName',''),

        # ── USAGE HISTORY ─────────────────────────────────────────────────────
        'usage_history_json':  json.dumps(to_json(
            attrs.get('usage_history', attrs.get('usageHistory',[]))) or []),
        'usage_daily_json':    json.dumps(to_json(attrs.get('usage_daily','')) or []),
        'usage_last_7d_json':  json.dumps(to_json(attrs.get('usage_last_7_days','')) or []),
        'usage_last_15d_json': json.dumps(to_json(attrs.get('usage_last_15_days','')) or []),

        # ── RAW BLOBS ─────────────────────────────────────────────────────────
        'raw_subsystems':      json.dumps(to_json(attrs.get('subsystems')) or {}),
        'raw_event_metadata':  json.dumps(to_json(attrs.get('eventMetadata')) or {}),
        'raw_dexter_config':   json.dumps(to_json(attrs.get('dexter_config')) or {}),
        'raw_access_control':  json.dumps(to_json(attrs.get('accessControl')) or {}),
    }

    # Event flags (including new ACS tamper)
    for col, attr_key in EVENT_FLAG_MAP.items():
        row[col] = attrs.get(attr_key)

    # Per-channel camera histories
    row.update(ch_dc)
    row.update(ch_tp)
    rows.append(row)

device_df = pd.DataFrame(rows)
print(f'\n✅ device_df: {len(device_df)} rows × {len(device_df.columns)} columns')
# Show coverage of new v11 columns
print('\n📊 New v11 column coverage:')
new_cols = ['dahua_model','dahua_cam_info','tailscale_ip','tailscale_hostname',
            'integrated_status','integrated_type','sw_id','sw_checksum',
            'gate_mili_time','cctv_mili_time','device_error','nvr_brand']
for col in new_cols:
    if col in device_df.columns:
        n   = device_df[col].replace('',None).replace('{}','').replace('[]','').dropna().shape[0]
        pct = 100*n//max(len(device_df),1)
        print(f'   {col:<25} {n:>5} ({pct}%)')


⚙️  Building master DataFrame for 184 devices ...

✅ device_df: 184 rows × 247 columns

📊 New v11 column coverage:
   dahua_model                   0 (0%)
   dahua_cam_info              184 (100%)
   tailscale_ip                 65 (35%)
   tailscale_hostname           65 (35%)
   integrated_status            60 (32%)
   integrated_type              58 (31%)
   sw_id                        65 (35%)
   sw_checksum                  65 (35%)
   gate_mili_time               59 (32%)
   cctv_mili_time               59 (32%)
   device_error                 60 (32%)
   nvr_brand                    59 (32%)


---
## Cell 10 — Fault Scoring Engine
### NEW v11: `integratedStatus` now scored (same weight as IAS)

In [10]:
def compute_gap_days(usage_json_str):
    try: history = json.loads(usage_json_str or '[]')
    except: return 0, None
    if not history: return 0, None
    try: history = sorted(history, key=lambda x: x.get('date',''))
    except: return 0, None
    max_s = cur = 0; ss = bs = None
    for e in history:
        try: cnt = float(e.get('count', e.get('value',-1)))
        except: cnt = -1
        if cnt == 0:
            cur += 1
            if cur == 1: ss = e.get('date','')
            if cur > max_s: max_s, bs = cur, ss
        else: cur = 0; ss = None
    return max_s, bs

def score_device(row):
    s, reasons = 0.0, []

    # Usage gap
    gap, gap_start = compute_gap_days(row.get('usage_history_json','[]'))
    if   gap >= 90: s += 50; reasons.append(f'GAP_{gap}d(+50)')
    elif gap >= 30: s += 40; reasons.append(f'GAP_{gap}d(+40)')
    elif gap >= 7:  s += 25; reasons.append(f'GAP_{gap}d(+25)')
    elif gap >= GAP_FAULT_DAYS: s += 12; reasons.append(f'GAP_{gap}d(+12)')

    # BAS downtime
    bd = safe_float(row.get('bas_downtime_min',0))
    if   bd >= 1440: s += 30; reasons.append(f'BAS_DT_{bd:.0f}m(+30)')
    elif bd >= 480:  s += 20; reasons.append(f'BAS_DT_{bd:.0f}m(+20)')
    elif bd >= 60:   s += 10; reasons.append(f'BAS_DT_{bd:.0f}m(+10)')

    # Zero data usage
    if safe_float(row.get('total_data_mb',-1)) == 0:
        s += 15; reasons.append('ZERO_DATA(+15)')

    # Inactive
    if row.get('inactive_since') and str(row['inactive_since']).strip() not in ('','null','None'):
        s += 20; reasons.append('INACTIVE(+20)')

    # NVR/DVR
    if is_fault(row.get('nvr_status')):
        s += 30; reasons.append(f'NVR={row["nvr_status"]}(+30)')
    elif is_active(row.get('ev_dvr_off')):
        s += 28; reasons.append('DVR_OFF(+28)')

    # HDD
    if is_fault(row.get('hdd_status')):
        s += 30; reasons.append(f'HDD={row["hdd_status"]}(+30)')
    elif is_active(row.get('ev_hdd_error')):
        s += 25; reasons.append('HDD_ERR(+25)')

    # Power / Gateway
    if is_active(row.get('ev_power_off')): s += 20; reasons.append('PWR_OFF(+20)')
    elif is_fault(row.get('gw_status')):   s += 15; reasons.append('GW_FAULT(+15)')

    # Camera disconnects
    dc = safe_int(row.get('cam_dc_count',0))
    if dc > 0: pts = min(dc*5,25); s += pts; reasons.append(f'CAM_DC={dc}(+{pts})')
    elif is_active(row.get('ev_cam_disconnect')): s += 15; reasons.append('CAM_DC_EV(+15)')

    # FAS
    if is_fault(row.get('fas_status')):
        s += 20; reasons.append('FAS_FAULT(+20)')
    elif is_active(row.get('ev_fas_off')): s += 20; reasons.append('FAS_OFF(+20)')
    elif is_active(row.get('ev_fas_fault')): s += 18; reasons.append('FAS_FLT_EV(+18)')

    # IAS
    if is_fault(row.get('ias_status')) or is_active(row.get('ev_ias_off')):
        s += 15; reasons.append('IAS_FAULT(+15)')

    # ── NEW v11: Integrated Alarm System scoring ───────────────────────────────
    if is_fault(row.get('integrated_status')):
        s += 15; reasons.append(f'INT_ALARM={row["integrated_status"]}(+15)')
    elif is_active(row.get('ev_int_off')):
        s += 15; reasons.append('INT_ALARM_OFF(+15)')

    # ACS / BAS / TLS
    if is_fault(row.get('acs_status')): s += 15; reasons.append('ACS_FAULT(+15)')
    if is_fault(row.get('bas_status')): s += 12; reasons.append('BAS_FAULT(+12)')
    if is_fault(row.get('tls_status')) or is_active(row.get('ev_tls_off')):
        s += 10; reasons.append('TLS_FAULT(+10)')

    # Battery
    if is_active(row.get('ev_battery_low')): s += 10; reasons.append('BATT_LOW(+10)')
    if is_active(row.get('ev_battery_reverse')): s += 8; reasons.append('BATT_REV(+8)')

    # SW state
    if str(row.get('sw_state','')).upper() in ('FAILED','FAILED_UPDATE','ERROR'):
        s += 10; reasons.append('FW_FAIL(+10)')

    # Disk / CPU
    disk = safe_float(row.get('disk_pct',0))
    cpu  = safe_float(row.get('cpu_pct',0))
    if   disk >= 90: s += 20; reasons.append(f'DISK_CRIT({disk:.0f}%)')
    elif disk >= 80: s += 15; reasons.append(f'DISK_HIGH({disk:.0f}%)')
    elif disk >= 75: s += 8;  reasons.append(f'DISK_WARN({disk:.0f}%)')
    if   cpu  >= 90: s += 10; reasons.append(f'CPU_HIGH({cpu:.0f}%)')

    score = round(min(s,100), 2)
    sev   = ('CRITICAL' if score>=70 else 'HIGH' if score>=45
             else 'MEDIUM' if score>=20 else 'HEALTHY')
    return score, sev, gap, gap_start or '', ' | '.join(reasons[:6]) or 'OK'

print('⚙️  Scoring ...')
device_df[['fault_score','severity','gap_days','gap_start','top_reasons']] = \
    device_df.apply(lambda r: pd.Series(score_device(r)), axis=1)
device_df = device_df.sort_values('fault_score', ascending=False).reset_index(drop=True)

print(f'\n✅ Scored {len(device_df)} devices\n')
counts = device_df['severity'].value_counts()
for sev in ['CRITICAL','HIGH','MEDIUM','HEALTHY']:
    n   = int(counts.get(sev,0))
    bar = '█' * (n*30//max(len(device_df),1))
    print(f'  {sev:<10} {n:>5}  {bar}')

# Show NVR brand breakdown
if 'nvr_brand' in device_df.columns:
    brand_counts = device_df['nvr_brand'].value_counts()
    print('\n📷 NVR Brand breakdown:')
    for brand, cnt in brand_counts.items():
        if brand: print(f'   {brand:<15} {cnt}')

# Show integrated alarm coverage
int_fault = (device_df['integrated_status'].apply(is_fault)).sum()
int_off   = device_df['ev_int_off'].apply(is_active).sum()
print(f'\n🚨 Integrated Alarm System:')
print(f'   Devices with FAULT status : {int_fault}')
print(f'   Devices with OFF event    : {int_off}')


⚙️  Scoring ...

✅ Scored 184 devices

  CRITICAL     109  █████████████████
  HIGH          35  █████
  MEDIUM         6  
  HEALTHY       34  █████

📷 NVR Brand breakdown:
   Hikvision       59

🚨 Integrated Alarm System:
   Devices with FAULT status : 8
   Devices with OFF event    : 6


---
## Cell 11 — Hierarchy Summary Tables

In [11]:
def agg_summary(group_cols):
    available = [c for c in group_cols if c in device_df.columns]
    return device_df.groupby(available, as_index=False, dropna=False).agg(
        devices           =('device_id',          'count'),
        avg_score         =('fault_score',          'mean'),
        max_score         =('fault_score',          'max'),
        critical          =('severity',             lambda x: (x=='CRITICAL').sum()),
        high              =('severity',             lambda x: (x=='HIGH').sum()),
        medium            =('severity',             lambda x: (x=='MEDIUM').sum()),
        healthy           =('severity',             lambda x: (x=='HEALTHY').sum()),
        gap_devices       =('gap_days',             lambda x: (x>=GAP_FAULT_DAYS).sum()),
        max_gap_days      =('gap_days',             'max'),
        total_data_mb     =('total_data_mb',        'sum'),
        bas_down_hrs      =('bas_downtime_min',     lambda x: round(x.sum()/60,1)),
        nvr_down_hrs      =('nvr_downtime_min',     lambda x: round(x.sum()/60,1)),
        fas_down_hrs      =('fas_downtime_min',     lambda x: round(x.sum()/60,1)),
        ias_down_hrs      =('ias_downtime_min',     lambda x: round(x.sum()/60,1)),
        cam_dc_total      =('cam_dc_count',         'sum'),
        # NEW v11: Dahua count, tailscale count
        dahua_devices     =('dahua_model',          lambda x: (x!='').sum()),
        tailscale_devices =('tailscale_ip',         lambda x: (x!='').sum()),
        integrated_faults =('integrated_status',    lambda x: x.apply(is_fault).sum()),
    ).assign(avg_score=lambda d: d['avg_score'].round(1)
    ).sort_values('avg_score', ascending=False)

bank_df   = agg_summary(['bank_name'])
ho_df     = agg_summary(['bank_name','ho_name'])
nbg_df    = agg_summary(['bank_name','nbg_name'])
zo_df     = agg_summary(['bank_name','nbg_name','zo_name'])
ro_df     = agg_summary(['bank_name','zo_name','ro_name'])
branch_df = agg_summary(['bank_name','nbg_name','zo_name','branch_name'])

bank_df['unique_zones']    = (device_df.groupby('bank_name')['zo_name']
                              .nunique().reindex(bank_df['bank_name']).values)
bank_df['unique_branches'] = (device_df.groupby('bank_name')['branch_name']
                              .nunique().reindex(bank_df['bank_name']).values)

print(f'✅ Summaries: bank={len(bank_df)} ho={len(ho_df)} nbg={len(nbg_df)}',
      f'zo={len(zo_df)} ro={len(ro_df)} branch={len(branch_df)}')
print('\nBANK RANKING:')
print(bank_df[['bank_name','unique_zones','unique_branches','devices',
               'avg_score','critical','high','dahua_devices',
               'tailscale_devices','integrated_faults','bas_down_hrs']]
      .to_string(index=False))


✅ Summaries: bank=22 ho=22 nbg=32 zo=41 ro=39 branch=136

BANK RANKING:
                bank_name  unique_zones  unique_branches  devices  avg_score  critical  high  dahua_devices  tailscale_devices  integrated_faults  bas_down_hrs
             Dexter 4 CUS             1                1        1      100.0         1     0              0                  1                  0           0.0
        DEXTER RANCHI CUS             1                1        1      100.0         1     0              0                  1                  0           0.0
            SDF-RASP5 CUS             1                1        1      100.0         1     0              0                  0                  0           0.0
             LOHARDAGA CC             1                1        1       85.0         1     0              0                  1                  0           0.0
           BANK OF BARODA             5               11       11       80.5        10     0              0                  2  

---
## Cell 12 — Key Discovery (v11 baseline for v11)

In [12]:
kc = {}
for a in device_data:
    for k,v in a.items():
        if not k.startswith('_') and v not in (None,'','[]','{}',{},[]):
            kc[k] = kc.get(k,0) + 1

kd = sorted(kc.items(), key=lambda x: -x[1])
known = set(CLIENT_KEYS + SERVER_KEYS + TELEMETRY_KEYS)
new_k = [(k,n) for k,n in kd
         if k not in known and not k.startswith('tele_') and not k.startswith('_')]

key_disc_df = pd.DataFrame(
    [(k, n, round(100*n/max(len(device_data),1),1)) for k,n in kd],
    columns=['key','devices_with_value','coverage_pct']
)

print(f'{len(kd)} unique keys | {len(new_k)} still undiscovered:')
for k,n in new_k[:20]:
    print(f'  {k:<55} {n:>5} ({100*n//max(len(device_data),1)}%)')
print('\n(Add any high-coverage keys to v11 CLIENT_KEYS or SERVER_KEYS)')


555 unique keys | 261 still undiscovered:
  total_data_mb                                             184 (100%)
  bas_downtime_min                                          184 (100%)
  audit_ts                                                  184 (100%)
  fault_score                                               184 (100%)
  fault_severity                                            184 (100%)
  fault_reasons                                             184 (100%)
  gap_days                                                  184 (100%)
  hierarchy_depth                                           184 (100%)
  full_path                                                 135 (73%)
  nbg_name                                                  134 (72%)
  bank_name                                                 130 (70%)
  zo_name                                                   129 (70%)
  branch_name                                               121 (65%)
  integrated_status                     

---
## Cell 13 — Export Excel (13 sheets) + Dashboard JSON + ML JSONL

In [13]:
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook
import os

ts         = datetime.now().strftime('%Y%m%d_%H%M')
XLSX_PATH  = f'tb_audit_v11_{ts}.xlsx'
JSON_PATH  = 'dashboard_data.json'
JSONL_PATH = 'ml_training_v11.jsonl'

FILLS = {'CRITICAL':PatternFill('solid',fgColor='FFCCCC'),
         'HIGH':    PatternFill('solid',fgColor='FFE5CC'),
         'MEDIUM':  PatternFill('solid',fgColor='FFFACC'),
         'HEALTHY': PatternFill('solid',fgColor='CCFFCC')}
HDR = PatternFill('solid',fgColor='1B3A5C')
HF  = Font(bold=True,color='FFFFFF')
HA  = Alignment(horizontal='center',vertical='center',wrap_text=True)

def style_ws(ws, sev_col=None):
    for c in ws[1]: c.fill=HDR; c.font=HF; c.alignment=HA
    ws.row_dimensions[1].height=28
    ws.freeze_panes='A2'
    for col in ws.columns:
        w=max((len(str(c.value or '')) for c in col),default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width=min(w+3,42)
    if sev_col:
        sc=next((c.column for c in ws[1] if str(c.value)==sev_col),None)
        if sc:
            for row in ws.iter_rows(min_row=2,min_col=sc,max_col=sc):
                for cell in row:
                    cell.fill=FILLS.get(str(cell.value),PatternFill())

# Drop raw/channel blob cols from main sheet
DROP = [c for c in device_df.columns
        if c.startswith('raw_') or c.startswith('camDC_') or c.startswith('camTP_')]
df_exp = device_df.drop(columns=DROP, errors='ignore')

# Hierarchy map sheet
hier_cols = ['bank_name','ho_name','nbg_name','lho_name','zo_name','ro_name',
             'co_name','rbo_name','branch_name','branch_code',
             'device_name','device_id','device_type','nvr_brand',
             'full_path','hierarchy_depth',
             'branch_lat','branch_lon','branch_city','branch_state',
             'tailscale_ip','tailscale_hostname',
             'install_date','go_live_date','contract_type','sla_tier']
hier_df = device_df[[c for c in hier_cols if c in device_df.columns]].copy()

# ── EXCEL ─────────────────────────────────────────────────────────────────────
with pd.ExcelWriter(XLSX_PATH, engine='openpyxl') as w:
    df_exp.to_excel(w,                                          sheet_name='All Devices',        index=False)
    df_exp[df_exp['fault_score']>=70].to_excel(w,               sheet_name='Critical',            index=False)
    df_exp[df_exp['gap_days']>0].sort_values('gap_days',ascending=False).to_excel(
                                                                w, sheet_name='Usage Gaps',        index=False)
    bank_df.to_excel(w,                                         sheet_name='Bank Summary',        index=False)
    ho_df.to_excel(w,                                           sheet_name='HO Summary',          index=False)
    nbg_df.to_excel(w,                                          sheet_name='NBG Summary',         index=False)
    zo_df.to_excel(w,                                           sheet_name='Zone Summary',        index=False)
    ro_df.to_excel(w,                                           sheet_name='RO Summary',          index=False)
    branch_df.to_excel(w,                                       sheet_name='Branch Summary',      index=False)
    hier_df.to_excel(w,                                         sheet_name='Hierarchy Map',       index=False)
    pd.DataFrame(customers).drop(columns=['_attrs'],errors='ignore').to_excel(
                                                                w, sheet_name='Customers (Banks)', index=False)
    pd.DataFrame([{k:v for k,v in a.items() if k!='_attrs'} for a in assets]
                ).to_excel(w,                                   sheet_name='Assets',              index=False)
    key_disc_df.to_excel(w,                                     sheet_name='Key Discovery',       index=False)

wb = load_workbook(XLSX_PATH)
SEV = {'All Devices','Critical','Usage Gaps'}
for sn in wb.sheetnames:
    style_ws(wb[sn], 'severity' if sn in SEV else None)
wb.save(XLSX_PATH)
print(f'✅ Excel: {XLSX_PATH} ({os.path.getsize(XLSX_PATH)//1024} KB) — {len(wb.sheetnames)} sheets')

# ── Dashboard JSON (v2 — full hierarchy tree, all devices, all levels) ────────────
def df2rec(d):
    return json.loads(d.to_json(orient='records', default_handler=str))

import math as _math

def _jval(v):
    if v is None: return None
    if isinstance(v, float):
        if _math.isnan(v) or _math.isinf(v): return None
        return round(v, 4) if not v.is_integer() else int(v)
    if hasattr(v, 'item'): return v.item()  # numpy scalar
    if isinstance(v, str) and v.strip().lower() in ('', 'nan', 'none', 'null', 'n/a', '-', 'na'):
        return None
    return v

def _agg_devs(devs):
    if not devs: return {'device_count': 0}
    scores = [d['fault_score'] for d in devs if d.get('fault_score') is not None]
    sevs = [str(d.get('severity', '')) for d in devs]
    return {
        'device_count':    len(devs),
        'avg_fault_score': round(sum(scores) / len(scores), 1) if scores else None,
        'max_fault_score': max(scores) if scores else None,
        'critical':  sum(1 for s in sevs if s == 'CRITICAL'),
        'high':      sum(1 for s in sevs if s == 'HIGH'),
        'medium':    sum(1 for s in sevs if s == 'MEDIUM'),
        'healthy':   sum(1 for s in sevs if s == 'HEALTHY'),
    }

print('   Building hierarchy tree (bank→HO→NBG→ZO→RO→branch) ...')
_SKIP_PFX = ('raw_', 'camDC_', 'camTP_')
tree = {}
for _, row in df_exp.iterrows():
    def _g(k): return str(row.get(k, '') or '').strip() or 'Unknown'
    bank = _g('bank_name'); ho = _g('ho_name'); nbg = _g('nbg_name')
    zo   = _g('zo_name');   ro = _g('ro_name'); branch = _g('branch_name')

    b  = tree.setdefault(bank,   {'_summary': {}, 'ho': {}})
    h  = b['ho'].setdefault(ho,  {'_summary': {}, 'nbg': {}})
    n  = h['nbg'].setdefault(nbg,{'_summary': {}, 'zo': {}})
    z  = n['zo'].setdefault(zo,  {'_summary': {}, 'ro': {}})
    r  = z['ro'].setdefault(ro,  {'_summary': {}, 'branches': {}})
    br = r['branches'].setdefault(branch, {'_summary': {}, 'devices': []})
    br['devices'].append(
        {k: _jval(v) for k, v in row.items()
         if not any(k.startswith(p) for p in _SKIP_PFX) and _jval(v) is not None}
    )

for bank, bdata in tree.items():
    bk_all = []
    for ho, hdata in bdata['ho'].items():
        ho_all = []
        for nbg, ndata in hdata['nbg'].items():
            nbg_all = []
            for zo, zdata in ndata['zo'].items():
                zo_all = []
                for ro, rdata in zdata['ro'].items():
                    ro_all = []
                    for branch, brdata in rdata['branches'].items():
                        brdata['_summary'] = _agg_devs(brdata['devices'])
                        ro_all += brdata['devices']
                    rdata['_summary'] = _agg_devs(ro_all); zo_all += ro_all
                zdata['_summary'] = _agg_devs(zo_all); nbg_all += zo_all
            ndata['_summary'] = _agg_devs(nbg_all); ho_all += nbg_all
        hdata['_summary'] = _agg_devs(ho_all); bk_all += ho_all
    bdata['_summary'] = _agg_devs(bk_all)

total_hos = sum(len(b['ho']) for b in tree.values())
print(f'   Tree: {len(tree)} banks, {total_hos} HOs, {len(device_df)} total devices')

dashboard = {
    'schema_version': 2,
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'summary': {
        'total_devices':  int(len(device_df)),
        'total_banks':    int(device_df['bank_name'].nunique()),
        'total_zones':    int(device_df['zo_name'].nunique() if 'zo_name' in device_df.columns else 0),
        'total_branches': int(device_df['branch_name'].nunique() if 'branch_name' in device_df.columns else 0),
        'critical': int((device_df['severity'] == 'CRITICAL').sum()),
        'high':     int((device_df['severity'] == 'HIGH').sum()),
        'medium':   int((device_df['severity'] == 'MEDIUM').sum()),
        'healthy':  int((device_df['severity'] == 'HEALTHY').sum()),
    },
    'hierarchy_tree': tree,
    'hierarchy_summaries': {
        'banks':    df2rec(bank_df),
        'ho':       df2rec(ho_df)     if 'ho_df'  in vars() else [],
        'nbg':      df2rec(nbg_df)    if 'nbg_df' in vars() else [],
        'zones':    df2rec(zo_df),
        'ro':       df2rec(ro_df)     if 'ro_df'  in vars() else [],
        'branches': df2rec(branch_df),
    },
}
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(dashboard, f, default=str, indent=2, ensure_ascii=False)
print(f'✅ Dashboard JSON: {JSON_PATH} ({os.path.getsize(JSON_PATH)//1024} KB)')

# ── ML JSONL (v11 — includes integrated_status, nvr_brand, tailscale) ─────────
EVENT_COLS  = [c for c in device_df.columns if c.startswith('ev_')]
STATUS_COLS = ['nvr_status','hdd_status','fas_status','ias_status','integrated_status',
               'acs_status','bas_status','tls_status','gw_status','cctv_status',
               'device_status','sw_state']

written = skipped = 0
with open(JSONL_PATH,'w',encoding='utf-8') as f:
    for _, row in device_df.iterrows():
        parts = []
        if row.get('bank_name'):       parts.append(f'bank:{row["bank_name"]}')
        if row.get('nbg_name'):        parts.append(f'nbg:{row["nbg_name"]}')
        if row.get('zo_name'):         parts.append(f'zo:{row["zo_name"]}')
        if row.get('branch_name'):     parts.append(f'branch:{row["branch_name"]}')
        if row.get('nvr_brand'):       parts.append(f'nvr_brand:{row["nvr_brand"]}')  # NEW
        gap = safe_int(row.get('gap_days',0))
        if gap > 0:                    parts.append(f'usage_gap_days:{gap}')
        for col in STATUS_COLS:
            v = str(row.get(col,'')).strip()
            if v and v.lower() not in ('','null','none','nan','0'):
                parts.append(f'{col}:{v}')
        active_evs = [c.replace('ev_','') for c in EVENT_COLS
                      if is_active(row.get(c))]
        if active_evs: parts.append(f'events:{",".join(active_evs)}')
        if safe_float(row.get('disk_pct',0)) > 0:
            parts.append(f'disk_pct:{row["disk_pct"]:.0f}')
        if safe_float(row.get('bas_downtime_min',0)) > 0:
            parts.append(f'bas_dt_min:{row["bas_downtime_min"]:.0f}')
        if safe_float(row.get('total_data_mb',0)) >= 0:
            parts.append(f'data_mb:{row["total_data_mb"]:.0f}')
        if safe_int(row.get('cam_dc_count',0)) > 0:
            parts.append(f'cam_dc:{row["cam_dc_count"]}')
        if row.get('tailscale_ip'):    parts.append('has_tailscale:1')  # NEW
        if row.get('hierarchy_depth'): parts.append(f'hier_depth:{row["hierarchy_depth"]}')
        if len(parts) < 3: skipped += 1; continue
        f.write(json.dumps({
            'input':  ' '.join(parts),
            'output': (f'fault_class:{row["severity"]} fault_score:{row["fault_score"]:.0f} '
                       f'gap_days:{gap} reasons:{row["top_reasons"]}'),
            '_meta':  {'device_id':row.get('device_id',''),
                       'bank':row.get('bank_name',''),
                       'nvr_brand':row.get('nvr_brand',''),
                       'score':row.get('fault_score',0)},
        }, ensure_ascii=False) + '\n')
        written += 1

print(f'✅ ML JSONL: {JSONL_PATH} — {written} examples ({skipped} skipped)')
print(f'\n📊 FINAL SUMMARY:')
print(f'   Devices    : {len(device_df)}')
print(f'   Banks      : {device_df["bank_name"].nunique()}')
print(f'   Zones      : {device_df["zo_name"].nunique()}')
print(f'   Branches   : {device_df["branch_name"].nunique()}')
print(f'   Columns    : {len(device_df.columns)}')
for sev in ['CRITICAL','HIGH','MEDIUM','HEALTHY']:
    print(f'   {sev:<10}: {int((device_df["severity"]==sev).sum())}')


✅ Excel: tb_audit_v11_20260529_1513.xlsx (380 KB) — 13 sheets
   Building hierarchy tree (bank→HO→NBG→ZO→RO→branch) ...
   Tree: 22 banks, 22 HOs, 184 total devices


✅ Dashboard JSON: dashboard_data.json (2284 KB)
✅ ML JSONL: ml_training_v11.jsonl — 171 examples (13 skipped)

📊 FINAL SUMMARY:
   Devices    : 184
   Banks      : 22
   Zones      : 20
   Branches   : 122
   Columns    : 252
   CRITICAL  : 109
   HIGH      : 35
   MEDIUM    : 6
   HEALTHY   : 34


---
## Cell 14 — Write Back to ThingsBoard (Optional)

In [14]:
from tqdm import tqdm as _tqdm
WRITE_BACK = True

if not WRITE_BACK:
    print('ℹ️  Skipped. Set WRITE_BACK = True.')
else:
    errors = []
    for _, row in _tqdm(device_df.iterrows(), total=len(device_df), desc='Writing'):
        dev_id = row['device_id']
        if not dev_id: continue
        payload = {
            'fault_score':       float(row['fault_score']),
            'fault_severity':    str(row['severity']),
            'fault_reasons':     str(row['top_reasons']),
            'gap_days':          int(row['gap_days']),
            'bas_downtime_min':  float(row.get('bas_downtime_min',0)),
            'total_data_mb':     float(row.get('total_data_mb',0)),
            'bank_name':         str(row.get('bank_name','')),
            'nbg_name':          str(row.get('nbg_name','')),
            'zo_name':           str(row.get('zo_name','')),
            'branch_name':       str(row.get('branch_name','')),
            'full_path':         str(row.get('full_path','')),
            'hierarchy_depth':   int(row.get('hierarchy_depth',0)),
            'nvr_brand':         str(row.get('nvr_brand','')),       # NEW
            'integrated_status': str(row.get('integrated_status','')),# NEW
            'audit_ts':          datetime.now().strftime('%Y-%m-%d %H:%M'),
        }
        url = f'{TB_HOST}/api/plugins/telemetry/DEVICE/{dev_id}/attributes/SERVER_SCOPE'
        try:
            r = session.post(url, headers=AUTH_HEADERS, json=payload, timeout=15)
            if r.status_code not in (200,201):
                errors.append(f'{row["device_name"]}: HTTP {r.status_code}')
        except Exception as e:
            errors.append(f'{row["device_name"]}: {e}')
        time.sleep(REQUEST_DELAY)
    print(f'Written: {len(device_df)-len(errors)}/{len(device_df)}')
    if errors: [print(f'  ⚠️  {e}') for e in errors[:10]]
    print('In TB: Device → Attributes → SERVER_SCOPE → fault_score / nvr_brand / integrated_status')


Writing:   0%|                                                                                 | 0/184 [00:00<?, ?it/s]

Writing:   1%|▍                                                                        | 1/184 [00:00<00:19,  9.41it/s]

Writing:   1%|▊                                                                        | 2/184 [00:00<00:19,  9.22it/s]

Writing:   2%|█▏                                                                       | 3/184 [00:00<00:20,  8.85it/s]

Writing:   2%|█▌                                                                       | 4/184 [00:00<00:21,  8.56it/s]

...

Writing:  99%|██████████████████████████████████████████████████████████████████████▏| 182/184 [00:20<00:00,  8.70it/s]

Writing:  99%|██████████████████████████████████████████████████████████████████████▌| 183/184 [00:20<00:00,  8.96it/s]

Writing: 100%|███████████████████████████████████████████████████████████████████████| 184/184 [00:20<00:00,  9.06it/s]

Writing: 100%|███████████████████████████████████████████████████████████████████████| 184/184 [00:20<00:00,  8.82it/s]

Written: 184/184
In TB: Device → Attributes → SERVER_SCOPE → fault_score / nvr_brand / integrated_status


---
## Cell 15 — Time-Series Config (365 Days)

These cells fetch **historical time-series** for every device going back 365 days.
Each day's snapshot = one ML training row.

**Expected output:**
- 184 devices × ~365 days = ~**67,000 training examples**
- `ts_device_df` — long-format DataFrame (one row per device per day)
- `ml_training_timeseries.jsonl` — full historical training file
- `ts_summary.xlsx` — daily snapshot Excel

**ThingsBoard API used:**
```
GET /api/plugins/telemetry/DEVICE/{id}/values/timeseries
    ?keys=key1,key2
    &startTs=<epoch_ms>
    &endTs=<epoch_ms>
    &limit=365
    &orderBy=ASC
    &agg=NONE
```

> ⏳ **Estimated time:** ~15–30 min for 184 devices × all keys


In [15]:
from datetime import datetime, timedelta
from tqdm import tqdm
import time

# ── Toggles ───────────────────────────────────────────────────────────────────
ENABLE_TS_KEY_DISCOVERY = True   # set False to skip TB discovery and use only
                                 # the hard-coded spec+legacy lists below

# ── Time range ────────────────────────────────────────────────────────────────
DAYS_BACK        = 365
TS_REQUEST_DELAY = 0.05
TS_LIMIT         = 5000
TS_AGG           = 'NONE'

now_ms   = int(datetime.utcnow().timestamp() * 1000)
start_ms = int((datetime.utcnow() - timedelta(days=DAYS_BACK)).timestamp() * 1000)

print(f'✅ Time-series config:')
print(f'   Period     : {DAYS_BACK} days')
print(f'   From       : {datetime.utcfromtimestamp(start_ms/1000).strftime("%Y-%m-%d %H:%M")} UTC')
print(f'   To         : {datetime.utcfromtimestamp(now_ms/1000).strftime("%Y-%m-%d %H:%M")} UTC')
print(f'   Limit/key  : {TS_LIMIT} points')
print(f'   Aggregation: {TS_AGG}')

# ══════════════════════════════════════════════════════════════════════════════
# CANONICAL TELEMETRY KEYS (spec: Dexter_HMS_Telemetry_Format_Spec)
# Groups A-H follow the spec sections 1-14. Group I holds legacy/tenant-specific
# keys that aren't in the spec but the previous harvest PROVED have data on this
# tenant (BAS_Downtime_Minutes, Total_Data_Usage, cpu/disk/temp, sw_state, etc.).
# ══════════════════════════════════════════════════════════════════════════════

# Group A — Event log (spec §1-2)
TS_KEYS_A_EVENTS = [
    'log_type', 'zone_no', 'date', 'time',
]
# Group B — Voltage & current (spec §3)
TS_KEYS_B_POWER = [
    'battery_voltage', 'ac_voltage', 'system_current',
]
# Group C — System status / statusbox (spec §4)
TS_KEYS_C_STATUSBOX = [
    'statusbox_system_on', 'statusbox_system_healthy', 'statusbox_mains_on',
    'statusbox_battery_reverse', 'statusbox_battery_low', 'statusbox_sos_status',
    'statusbox_network', 'statusbox_no_of_connected_device',
]
# Group D — Heartbeats (spec §5)
TS_KEYS_D_HEARTBEAT = [
    'heartbeat_BAS', 'heartbeat_FAS', 'heartbeat_CCTV', 'heartbeat_IBAS',
    'heartbeat_access_control', 'heartbeat_time_lock',
]
# Group E — SD card info (spec §6-8). JSON-valued; parsed in Cell 17.
TS_KEYS_E_SDCARD = [
    'Hik_SD_card_info', 'Dahua_SD_card_info', 'Cpplus_SD_card_info',
]
# Group F — Texecom (spec §9-12). JSON-valued.
TS_KEYS_F_TEXECOM = [
    'texecom_heartbeat', 'texecom_power_state', 'texecom_event', 'texecom_panel_info',
]
# Group G — GPS (spec §13)
TS_KEYS_G_GPS = ['lat', 'lon']
# Group H — Tailscale VPN (spec §14). JSON-valued.
TS_KEYS_H_TAILSCALE = ['tailscale_data']
# Group I — Legacy / tenant-extension keys
TS_KEYS_I_LEGACY = [
    'BAS_Downtime_Minutes', 'BAS_Uptime_Minutes',
    'Total_Data_Usage', 'cavlidata_ontime',
    'cpu', 'disk', 'memory', 'temperature', 'frequency',
    'net_recv_mb', 'net_sent_mb', 'rpi_usage',
    'sw_state', 'target_sw_tag', 'target_sw_title', 'target_sw_ts', 'target_sw_version',
    'arrLat', 'arrLon',
    'heartBeatBAS', 'heartBeatFAS', 'heartBeatCCTV', 'heartBeatTL',
]

SPEC_GROUPS = {
    'A:events':    TS_KEYS_A_EVENTS,
    'B:power':     TS_KEYS_B_POWER,
    'C:statusbox': TS_KEYS_C_STATUSBOX,
    'D:heartbeat': TS_KEYS_D_HEARTBEAT,
    'E:sdcard':    TS_KEYS_E_SDCARD,
    'F:texecom':   TS_KEYS_F_TEXECOM,
    'G:gps':       TS_KEYS_G_GPS,
    'H:tailscale': TS_KEYS_H_TAILSCALE,
    'I:legacy':    TS_KEYS_I_LEGACY,
}
SPEC_AND_LEGACY_KEYS = [k for g in SPEC_GROUPS.values() for k in g]
_seen = set()
SPEC_AND_LEGACY_KEYS = [k for k in SPEC_AND_LEGACY_KEYS
                        if not (k in _seen or _seen.add(k))]

# JSON-valued keys — parsed in Cell 17
JSON_VALUE_KEYS = set(TS_KEYS_E_SDCARD + TS_KEYS_F_TEXECOM + TS_KEYS_H_TAILSCALE)

print(f'\n   Spec + legacy keys : {len(SPEC_AND_LEGACY_KEYS)}')
for lab, g in SPEC_GROUPS.items():
    print(f'      {lab:<14} : {len(g):>2} keys')


# ══════════════════════════════════════════════════════════════════════════════
# AUTO-DISCOVERY — fetch the actual TS-key list per device from TB and union it
# in. This catches NEW keys added after this script was written without code
# changes. Toggle with ENABLE_TS_KEY_DISCOVERY at the top of this cell.
# ══════════════════════════════════════════════════════════════════════════════

DISCOVERED_TS_KEYS = []
new_keys_from_tb   = []

if ENABLE_TS_KEY_DISCOVERY:
    print(f'\n🔍 Auto-discovering telemetry keys for {len(all_devices)} devices ...')
    discovered = set()
    discovery_errors = 0
    for d in tqdm(all_devices, desc='Discovery', unit='device'):
        dev_id = d.get('id', {}).get('id', '')
        if not dev_id: continue
        url = f'{TB_HOST}/api/plugins/telemetry/DEVICE/{dev_id}/keys/timeseries'
        try:
            r = session.get(url, headers=AUTH_HEADERS, timeout=15)
            if r.status_code == 200:
                for k in (r.json() or []):
                    if isinstance(k, str): discovered.add(k)
            elif r.status_code == 401:
                raise Exception('JWT expired — re-run Cell 3')
            else:
                discovery_errors += 1
        except Exception:
            discovery_errors += 1
        time.sleep(0.02)

    DISCOVERED_TS_KEYS = sorted(discovered)
    known = set(SPEC_AND_LEGACY_KEYS)
    new_keys_from_tb = [k for k in DISCOVERED_TS_KEYS if k not in known]
    in_spec_count    = len(DISCOVERED_TS_KEYS) - len(new_keys_from_tb)
    in_spec_missing  = [k for k in SPEC_AND_LEGACY_KEYS if k not in discovered]

    print(f'\n   Devices queried  : {len(all_devices)}')
    print(f'   Discovery errors : {discovery_errors}')
    print(f'   Keys discovered  : {len(DISCOVERED_TS_KEYS)}')
    print(f'   Already in spec  : {in_spec_count}')
    print(f'   NEW (not in spec): {len(new_keys_from_tb)}')

    if new_keys_from_tb:
        print(f'\n   📌 NEW keys discovered in TB (auto-merged into ALL_TS_KEYS):')
        for k in new_keys_from_tb[:50]:
            print(f'      + {k}')
        if len(new_keys_from_tb) > 50:
            print(f'      ... and {len(new_keys_from_tb)-50} more')
        print(f'\n   👉 If any of these need custom scoring/JSON parsing,')
        print(f'      add them to Cells 17 (parsing) and 18 (scoring).')

    if in_spec_missing:
        print(f'\n   ℹ  {len(in_spec_missing)} spec/legacy keys not seen on this tenant')
        print(f'      (still fetched, will show 0% coverage — likely subsystem absent)')
else:
    print('\n   ℹ  Auto-discovery disabled — using hard-coded spec/legacy list only.')


# ── Final unified key list ────────────────────────────────────────────────────
ALL_TS_KEYS = SPEC_AND_LEGACY_KEYS + new_keys_from_tb

# Re-slice into HTTP request batches (≤25 keys per request to stay under URL length)
REQUEST_BATCH_SIZE = 25
ALL_TS_KEY_GROUPS  = [
    ALL_TS_KEYS[i:i+REQUEST_BATCH_SIZE]
    for i in range(0, len(ALL_TS_KEYS), REQUEST_BATCH_SIZE)
]

print(f'\n✅ Final TS-key plan:')
print(f'   Total TS keys      : {len(ALL_TS_KEYS)}')
print(f'   HTTP batches       : {len(ALL_TS_KEY_GROUPS)} (≤{REQUEST_BATCH_SIZE}/batch)')
print(f'   Estimated requests : {len(all_devices)} × {len(ALL_TS_KEY_GROUPS)} = '
      f'{len(all_devices) * len(ALL_TS_KEY_GROUPS)} calls')


✅ Time-series config:
   Period     : 365 days
   From       : 2025-05-29 04:13 UTC
   To         : 2026-05-29 04:13 UTC
   Limit/key  : 5000 points
   Aggregation: NONE

   Spec + legacy keys : 54
      A:events       :  4 keys
      B:power        :  3 keys
      C:statusbox    :  8 keys
      D:heartbeat    :  6 keys
      E:sdcard       :  3 keys
      F:texecom      :  4 keys
      G:gps          :  2 keys
      H:tailscale    :  1 keys
      I:legacy       : 23 keys

🔍 Auto-discovering telemetry keys for 184 devices ...


Discovery:   0%|                                                                           | 0/184 [00:00<?, ?device/s]

Discovery:   1%|▋                                                                  | 2/184 [00:00<00:12, 14.57device/s]

Discovery:   2%|█▍                                                                 | 4/184 [00:00<00:12, 14.95device/s]

Discovery:   3%|██▏                                                                | 6/184 [00:00<00:12, 14.77device/s]

...

Discovery:  98%|███████████████████████████████████████████████████████████████▌ | 180/184 [00:12<00:00, 14.79device/s]

Discovery:  99%|████████████████████████████████████████████████████████████████▎| 182/184 [00:12<00:00, 15.13device/s]

Discovery: 100%|█████████████████████████████████████████████████████████████████| 184/184 [00:12<00:00, 15.00device/s]

Discovery: 100%|█████████████████████████████████████████████████████████████████| 184/184 [00:12<00:00, 14.48device/s]


   Devices queried  : 184
   Discovery errors : 0
   Keys discovered  : 593
   Already in spec  : 52
   NEW (not in spec): 541

   📌 NEW keys discovered in TB (auto-merged into ALL_TS_KEYS):
      + ACSheartbeatCount
      + ACSheartbeatTS
      + ACSofftimeTS
      + ACfaultCOUNT
      + ACinactiveCOUNT
      + AntiDismantleStatus
      + AutoDialer
      + AutoDialerUptime
      + BACSsyncDateTime
      + BAS_Downtime_Hours
      + BAS_Expected_Heartbeats
      + BAS_Hang_Events
      + BAS_Received_Heartbeats
      + BAS_Uptime_%
      + BAS_Uptime_Hours
      + BASfaultCOUNT
      + BASheartbeatCount
      + BASheartbeatTS
      + BASheartbeat_Duration
      + BASinactiveCOUNT
      + BASofftimeTS
      + BASofftime_Duration
      + CAMERA_DETAILS
      + CAMERAdETAILS
      + CCTVheartbeatCount
      + CCTVheartbeatTS
      + CCTVofftimeTS
      + CPPLUS_recordings
      + CPPlus_NVR_model
      + CP_PlusNVR_Heartbeat
      + CP_Plus_NVR_CameraRecInfo
      + CP_Plus_NVR_Date
   

---
## Cell 16 — Fetch Historical Time-Series per Device

Makes `len(devices) × len(groups)` API calls.
Results stored in `raw_ts` — dict of `device_id → {key → [(ts, value), ...]}`


In [16]:
from tqdm import tqdm

def fetch_ts_group(device_id, keys, start_ms, end_ms, limit=365, agg='NONE'):
    """
    Fetch time-series for a list of keys for one device.
    Returns dict: { key: [ {ts: epoch_ms, value: ...}, ... ] }
    Handles pagination automatically if limit > 1000.
    """
    result   = {}
    keys_str = ','.join(keys)
    url = (
        f'{TB_HOST}/api/plugins/telemetry/DEVICE/{device_id}/values/timeseries'
        f'?keys={keys_str}'
        f'&startTs={start_ms}'
        f'&endTs={end_ms}'
        f'&limit={limit}'
        f'&orderBy=ASC'
        f'&agg={agg}'
        f'&useStrictDataTypes=false'
    )
    try:
        r = session.get(url, headers=AUTH_HEADERS, timeout=30)
        if r.status_code == 200:
            for key, entries in r.json().items():
                result[key] = entries or []
        elif r.status_code == 401:
            raise Exception('JWT expired — re-run Cell 3')
        else:
            result['_error'] = f'HTTP {r.status_code}'
    except Exception as e:
        result['_error'] = str(e)
    return result


# ── Main fetch loop ────────────────────────────────────────────────────────────
print(f'📡 Fetching 365-day time-series for {len(all_devices)} devices ...')
print(f'   {len(ALL_TS_KEY_GROUPS)} API calls per device = {len(all_devices)*len(ALL_TS_KEY_GROUPS)} total calls')
print(f'   Estimated time: {len(all_devices)*len(ALL_TS_KEY_GROUPS)*TS_REQUEST_DELAY/60:.1f} min (without server latency)\n')

raw_ts    = {}   # device_id → { key → list of {ts, value} }
ts_errors = []

for device in tqdm(all_devices, desc='TS Fetch', unit='device'):
    dev_id   = device.get('id',{}).get('id','')
    dev_name = device.get('name','UNKNOWN')
    dev_ts   = {}

    for group in ALL_TS_KEY_GROUPS:
        try:
            chunk = fetch_ts_group(
                dev_id, group,
                start_ms, now_ms,
                limit=TS_LIMIT, agg=TS_AGG
            )
            for key, entries in chunk.items():
                if key != '_error':
                    dev_ts[key] = entries
                else:
                    ts_errors.append(f'{dev_name}: {entries}')
        except Exception as e:
            ts_errors.append(f'{dev_name}: {e}')
        time.sleep(TS_REQUEST_DELAY)

    raw_ts[dev_id] = dev_ts

# Summary
total_points = sum(
    len(entries)
    for dev in raw_ts.values()
    for entries in dev.values()
)
print(f'\n✅ Time-series fetched:')
print(f'   Devices     : {len(raw_ts)}')
print(f'   Total points: {total_points:,}')
print(f'   Errors       : {len(ts_errors)}')
if ts_errors:
    for e in ts_errors[:5]: print(f'   ⚠️  {e}')

# Show sample for first device
sample_dev_id = list(raw_ts.keys())[0]
sample_name   = all_devices[0].get('name','?')
print(f'\nSample — {sample_name}:')
for key, entries in list(raw_ts[sample_dev_id].items())[:6]:
    n = len(entries)
    if n > 0:
        first_ts = epoch_ms(entries[0]['ts'])
        last_ts  = epoch_ms(entries[-1]['ts'])
        print(f'   {key:<30} {n:>4} points  [{first_ts} → {last_ts}]')


📡 Fetching 365-day time-series for 184 devices ...
   24 API calls per device = 4416 total calls
   Estimated time: 3.7 min (without server latency)



TS Fetch:   0%|                                                                            | 0/184 [00:00<?, ?device/s]

TS Fetch:   1%|▎                                                                   | 1/184 [00:02<07:53,  2.59s/device]

TS Fetch:   1%|▋                                                                   | 2/184 [00:16<28:19,  9.34s/device]

TS Fetch:   2%|█                                                                   | 3/184 [00:19<18:58,  6.29s/device]

...

TS Fetch:  99%|█████████████████████████████████████████████████████████████████▎| 182/184 [41:57<00:06,  3.43s/device]

TS Fetch:  99%|█████████████████████████████████████████████████████████████████▋| 183/184 [42:00<00:03,  3.24s/device]

TS Fetch: 100%|██████████████████████████████████████████████████████████████████| 184/184 [42:16<00:00,  7.13s/device]

TS Fetch: 100%|██████████████████████████████████████████████████████████████████| 184/184 [42:16<00:00, 13.79s/device]


✅ Time-series fetched:
   Devices     : 184
   Total points: 25,855,456
   Errors       : 17
   ⚠️  BOI-AIZWAL: HTTP 400
   ⚠️  BOI-BALLYBAZAR: HTTP 400
   ⚠️  BOI-BARIPADA-CC: HTTP 400
   ⚠️  BOI-BERHAMPUR: HTTP 400
   ⚠️  BOI-DANKUNI: HTTP 400

Sample — abc:
   log_type                         25 points  [2026-04-21 04:39 → 2026-04-21 05:38]
   date                             25 points  [2026-04-21 04:39 → 2026-04-21 05:38]
   time                             25 points  [2026-04-21 04:39 → 2026-04-21 05:38]
   lat                               2 points  [2026-04-21 04:39 → 2026-04-21 05:26]
   lon                               2 points  [2026-04-21 04:39 → 2026-04-21 05:26]
   BAS_Downtime_Minutes              1 points  [2026-03-23 00:00 → 2026-03-23 00:00]


---
## Cell 17 — Align All Keys to Daily Snapshots

ThingsBoard stores each key independently with its own timestamps.
This cell aligns everything to a **daily grid** (one row per device per day)
using forward-fill — i.e. a key's last known value carries forward until a new reading arrives.

**Output:** `ts_device_df` — long-format DataFrame
```
device_id | date       | bank | nbg | zo | branch | nvrStatus | hddStatus | ... | fault_label
dev001    | 2025-05-13 | BOI  | ... | ...| ...    | Healthy   | Healthy   | ... | CRITICAL
dev001    | 2025-05-14 | BOI  | ... | ...| ...    | OFFLINE   | Healthy   | ... | CRITICAL
```


In [17]:
import pandas as pd
import json as _json
import re
import math
from datetime import datetime, timedelta

# ── Build date grid ────────────────────────────────────────────────────────────
date_range = pd.date_range(
    start = datetime.utcnow() - timedelta(days=DAYS_BACK),
    end   = datetime.utcnow(),
    freq  = 'D'
)
date_strs = [d.strftime('%Y-%m-%d') for d in date_range]
print(f'Daily grid: {len(date_strs)} days  ({date_strs[0]} → {date_strs[-1]})')


# ── Helpers ───────────────────────────────────────────────────────────────────
_MISSING_STR = {'','none','null','nan','n/a','na','unknown','undefined','-'}

def is_missing(v):
    """True if v is None / empty / 'N/A' / 'unknown' etc.  Not a fault."""
    if v is None: return True
    if isinstance(v, float):
        try:
            if math.isnan(v): return True
        except: pass
        return False
    if isinstance(v, str):
        return v.strip().lower() in _MISSING_STR
    return False

def _parse_json(v):
    if v is None: return None
    if isinstance(v, (dict, list)): return v
    if isinstance(v, str):
        s = v.strip()
        if s.startswith('{') or s.startswith('['):
            try: return _json.loads(s)
            except: return None
    return None

def _extract_data_usage(v):
    """Total_Data_Usage value may be JSON list like [{"Date":...,"usage":N}, ...]
    Return the latest numeric usage if parseable, else best-effort float, else None."""
    if v is None: return None
    if isinstance(v, (int, float)):
        try:
            return None if math.isnan(v) else float(v)
        except: return None
    parsed = _parse_json(v)
    if isinstance(parsed, list) and parsed:
        # Find last numeric value in last entry
        last = parsed[-1]
        if isinstance(last, dict):
            for key in ('usage','usage_mb','data_mb','total','value','Usage'):
                if key in last:
                    try: return float(last[key])
                    except: pass
            # Fallback: any numeric value
            for val in last.values():
                try: return float(val)
                except: pass
        else:
            try: return float(last)
            except: pass
        return None
    if isinstance(parsed, dict):
        for key in ('usage','usage_mb','data_mb','total','value'):
            if key in parsed:
                try: return float(parsed[key])
                except: pass
        return None
    # Plain string number?
    try: return float(str(v).strip())
    except: return None


def ts_entries_to_daily(entries, date_strs):
    if not entries:
        return {d: None for d in date_strs}
    sorted_pts = sorted(entries, key=lambda x: x['ts'])
    day_map = {}
    for pt in sorted_pts:
        day = datetime.utcfromtimestamp(pt['ts']/1000).strftime('%Y-%m-%d')
        day_map[day] = pt['value']
    out, last = {}, None
    for d in date_strs:
        if d in day_map: last = day_map[d]
        out[d] = last
    return out

def entries_per_day_count(entries, date_strs, predicate=None):
    out = {d: 0 for d in date_strs}
    if not entries: return out
    for pt in entries:
        day = datetime.utcfromtimestamp(pt['ts']/1000).strftime('%Y-%m-%d')
        if day not in out: continue
        if predicate is None or predicate(pt.get('value')):
            out[day] += 1
    return out

def entries_change_count(entries, date_strs):
    """Per-day count of value transitions (value differs from previous reading)."""
    out = {d: 0 for d in date_strs}
    if not entries: return out
    sorted_pts = sorted(entries, key=lambda x: x['ts'])
    prev = None
    for pt in sorted_pts:
        day = datetime.utcfromtimestamp(pt['ts']/1000).strftime('%Y-%m-%d')
        if day not in out: continue
        v = pt.get('value')
        if prev is not None and v != prev:
            out[day] += 1
        prev = v
    return out


FAULT_EVENT_TYPES = {
    'power_off','power_cut','battery_low','battery_reverse',
    'network_disconnect',
    'intrusion_alarm_system_activate','intrusion_alarm_system_fault',
    'fire_alarm_system_activate','fire_alarm_system_fault',
    'integrated_alarm_system_activate','integrated_alarm_system_fault',
    'access_control_system_tamper',
    'time_lock_tamper',
    'dvr_nvr_off','camera_disconnect','camera_tampered','hdd_error',
}


# ── Test-device filter ────────────────────────────────────────────────────────
# Devices we recognise as synthetic/internal. Kept in CSV with is_real_device=False,
# excluded from Top Risk Days (Cell 19). Pattern adjustable.
_TEST_NAME_RE = re.compile(
    r'^(asset[\s_-]*test|prod\d*[\s_-]*sm?\d*|sdf[-_]|dexter\d+|slink[-_])',
    re.IGNORECASE,
)
def is_real_device(name, bank):
    if name is None: return False
    if _TEST_NAME_RE.match(str(name)): return False
    if bank is None or str(bank).strip().lower() in _MISSING_STR: return False
    return True


# ── Hierarchy + SERVER-ATTRIBUTE SNAPSHOT lookup from device_df ───────────────
SNAPSHOT_COLS = [
    'bank_name','ho_name','nbg_name','lho_name','zo_name','ro_name',
    'co_name','rbo_name','branch_name','branch_code',
    'full_path','hierarchy_depth','device_name',
    'device_type','customer_name','nvr_brand',
    'nvr_status','hdd_status','fas_status','ias_status','bas_status',
    'acs_status','tls_status','gw_status','cctv_status','power_status',
    'integrated_status','recording_status','ups_status',
    'cam_total','cam_online','cam_offline','cam_dc_count','cam_tamper_count',
    'count_ch','count_hdd',
]
_avail_cols = [c for c in SNAPSHOT_COLS if c in device_df.columns]
hier_lookup = (
    device_df.set_index('device_id')[_avail_cols].to_dict(orient='index')
    if 'device_id' in device_df.columns else {}
)

_SNAPSHOT_SUFFIX_COLS = {
    'nvr_status','hdd_status','fas_status','ias_status','bas_status',
    'acs_status','tls_status','gw_status','cctv_status','power_status',
    'integrated_status',
    'cam_online','cam_offline','cam_dc_count','cam_tamper_count',
}

# Collision guard
RESERVED_COL_NAMES = {
    'device_id','date','time',
    'bank_name','ho_name','nbg_name','lho_name','zo_name','ro_name',
    'co_name','rbo_name','branch_name','branch_code',
    'full_path','hierarchy_depth','device_name',
    'device_type','customer_name','nvr_brand',
    'nbg','zo','branch',
    'id','status','attribute','alarm',
}
def _safe_colname(k):
    return f'ts_{k}' if k in RESERVED_COL_NAMES else k
_TS_COLNAME = {k: _safe_colname(k) for k in ALL_TS_KEYS}


# ── Build long-format rows ─────────────────────────────────────────────────────
print(f'\n⚙️  Aligning {len(raw_ts)} devices × {len(date_strs)} days ...')

ts_rows = []
n_real = n_test = 0
for dev_id, dev_ts in tqdm(raw_ts.items(), desc='Aligning', unit='device'):
    daily = {k: ts_entries_to_daily(dev_ts.get(k, []), date_strs) for k in ALL_TS_KEYS}

    # Event counts from log_type
    log_entries = dev_ts.get('log_type', [])
    daily_fault_evt = entries_per_day_count(
        log_entries, date_strs,
        predicate=lambda v: str(v or '').strip().lower() in FAULT_EVENT_TYPES,
    )
    daily_all_evt = entries_per_day_count(log_entries, date_strs)

    h = hier_lookup.get(dev_id, {})
    dev_name = h.get('device_name', '')
    dev_bank = h.get('bank_name', '')
    real     = is_real_device(dev_name, dev_bank)
    if real: n_real += 1
    else:    n_test += 1

    for date_str in date_strs:
        row = {'device_id': dev_id, 'date': date_str, 'is_real_device': real}
        for c in _avail_cols:
            colname = c + ('_snapshot' if c in _SNAPSHOT_SUFFIX_COLS else '')
            row[colname] = h.get(c, '')

        for k in ALL_TS_KEYS:
            v = daily[k][date_str]
            if isinstance(v, (dict, list)):
                try: v = _json.dumps(v, ensure_ascii=False)[:32000]
                except: v = str(v)[:32000]
            row[_TS_COLNAME[k]] = v

        # Total_Data_Usage → parsed numeric, replaces stringified JSON
        if 'Total_Data_Usage' in _TS_COLNAME:
            row['Total_Data_Usage_num'] = _extract_data_usage(
                row.get(_TS_COLNAME['Total_Data_Usage'])
            )

        row['log_event_count'] = daily_all_evt.get(date_str, 0)
        row['log_fault_count'] = daily_fault_evt.get(date_str, 0)

        # SD card parsed summaries
        for jk, prefix in [
            ('Hik_SD_card_info','hik_sd'),
            ('Dahua_SD_card_info','dahua_sd'),
            ('Cpplus_SD_card_info','cpplus_sd'),
        ]:
            raw_v = row.get(_TS_COLNAME.get(jk, jk))
            parsed = _parse_json(raw_v)
            n_total = n_bad = 0
            free_gb = used_gb = 0.0
            if isinstance(parsed, list):
                for entry in parsed:
                    if not isinstance(entry, dict): continue
                    n_total += 1
                    if str(entry.get('sd_status','')).strip().lower() in (
                        'error','fault','full','missing','bad','offline','0','false'):
                        n_bad += 1
                    try: free_gb += float(entry.get('free_space',0))/1024
                    except: pass
                    try: used_gb += float(entry.get('used_space',0))/1024
                    except: pass
            elif isinstance(parsed, dict):
                n_total = 1
                if str(parsed.get('sd_status','')).strip().lower() in (
                    'error','fault','full','missing','bad','offline','0','false'):
                    n_bad = 1
            row[f'{prefix}_cam_count'] = n_total
            row[f'{prefix}_bad_count'] = n_bad

        ts_rows.append(row)

ts_device_df = pd.DataFrame(ts_rows)

print(f'\n✅ ts_device_df built:')
print(f'   Rows               : {len(ts_device_df):,}')
print(f'   Columns            : {len(ts_device_df.columns)}')
print(f'   Devices            : {ts_device_df["device_id"].nunique()}')
print(f'   Real devices       : {n_real}')
print(f'   Test/synthetic     : {n_test}  (kept in CSV, excluded from Top Risk)')
print(f'   Date range         : {ts_device_df["date"].min()} → {ts_device_df["date"].max()}')

# Quick coverage report (sorted)
print('\n📊 TS key coverage (sorted, top 60):')
total_rows = len(ts_device_df)
cov = []
for k in ALL_TS_KEYS:
    col = _TS_COLNAME[k]
    if col in ts_device_df.columns:
        s = ts_device_df[col].astype(str).str.strip()
        n = (~s.str.lower().isin(_MISSING_STR)).sum()
        pct = 100 * n // max(total_rows, 1)
        cov.append((k, col, pct))
cov.sort(key=lambda x: -x[2])
for k, col, pct in cov[:60]:
    bar = '█' * (pct // 5)
    print(f'    {k:<38} {pct:>4}%  {bar}')
print(f'\n   {sum(1 for _,_,p in cov if p>0)} keys have data, {sum(1 for _,_,p in cov if p==0)} keys at 0%')


Daily grid: 366 days  (2025-05-29 → 2026-05-29)

⚙️  Aligning 184 devices × 366 days ...


Aligning:   0%|                                       | 0/184 [00:00<?, ?device/s]

Aligning:   1%|▏                              | 1/184 [00:00<00:24,  7.36device/s]

Aligning:   1%|▎                              | 2/184 [00:00<01:11,  2.56device/s]

Aligning:   2%|▌                              | 3/184 [00:00<00:49,  3.64device/s]

...

Aligning:  99%|████████████████████████████▊| 183/184 [02:29<00:00,  6.64device/s]

Aligning: 100%|█████████████████████████████| 184/184 [02:30<00:00,  4.38device/s]

Aligning: 100%|█████████████████████████████| 184/184 [02:30<00:00,  1.23device/s]


✅ ts_device_df built:
   Rows               : 67,344
   Columns            : 643
   Devices            : 184
   Real devices       : 128
   Test/synthetic     : 56  (kept in CSV, excluded from Top Risk)
   Date range         : 2025-05-29 → 2026-05-29

📊 TS key coverage (sorted, top 60):


    heartbeat_BAS                           100%  ████████████████████
    heartbeat_FAS                           100%  ████████████████████
    heartbeat_CCTV                          100%  ████████████████████
    heartbeat_IBAS                          100%  ████████████████████
    heartbeat_access_control                100%  ████████████████████
    heartbeat_time_lock                     100%  ████████████████████
    Hik_SD_card_info                        100%  ████████████████████
    Dahua_SD_card_info                      100%  ████████████████████
    Cpplus_SD_card_info                     100%  ████████████████████
    texecom_heartbeat                       100%  ████████████████████
    texecom_power_state                     100%  ████████████████████
    texecom_event                           100%  ████████████████████
    texecom_panel_info                      100%  ████████████████████
    tailscale_data                          100%  ████████████████████
    BA

---
## Cell 18 — Label Each Day with Fault Score

Run the same scoring engine on each historical daily snapshot.
This gives every row a `fault_score`, `severity`, and `top_reasons`.

**This is what turns your time-series data into ML training labels.**


In [18]:
def is_fault_val(v):
    """True ONLY for explicit fault sentinels — NOT for missing/N/A."""
    if v is None: return False
    s = str(v).strip().upper()
    if s in _MISSING_STR_UP: return False         # missing != fault
    return s in {
        'OFFLINE','OFF','FAULT','ERROR','INACTIVE',
        'DISCONNECTED','DOWN','FAILED','FAILED_UPDATE','BAD','FALSE','0',
    }
_MISSING_STR_UP = {s.upper() for s in _MISSING_STR}

def is_offline(v):
    if v is None: return False
    s = str(v).strip().lower()
    if s in _MISSING_STR: return False             # missing != offline
    return s in ('offline','off','down','disconnected','0','false')

def _bool_false(v):
    if v is None or is_missing(v): return False
    return str(v).strip().lower() in ('false','0','no','off')

def _bool_true(v):
    if v is None or is_missing(v): return False
    return str(v).strip().lower() in ('true','1','yes','on','active')


def score_ts_row(row):
    s, reasons = 0.0, []

    # ── Statusbox (spec §4) — skip when missing ──────────────────────────────
    if _bool_false(row.get('statusbox_system_on')):
        s += 35; reasons.append('SYS_OFF(+35)')
    if _bool_false(row.get('statusbox_system_healthy')):
        s += 25; reasons.append('SYS_UNHEALTHY(+25)')
    if _bool_false(row.get('statusbox_mains_on')):
        s += 20; reasons.append('MAINS_OFF(+20)')
    if _bool_true(row.get('statusbox_battery_low')):
        s += 15; reasons.append('BATT_LOW(+15)')
    if _bool_true(row.get('statusbox_battery_reverse')):
        s += 10; reasons.append('BATT_REVERSE(+10)')
    if _bool_true(row.get('statusbox_sos_status')):
        s += 20; reasons.append('SOS_ACTIVE(+20)')

    # ── Heartbeats (spec §5 + legacy CamelCase) ──────────────────────────────
    HB_WEIGHT = {
        'heartbeat_BAS':15,'heartbeat_FAS':15,'heartbeat_CCTV':12,
        'heartbeat_IBAS':12,'heartbeat_access_control':12,'heartbeat_time_lock':10,
    }
    for hb_key, pts in HB_WEIGHT.items():
        if is_offline(row.get(hb_key)):
            s += pts; reasons.append(f'{hb_key}_OFFLINE(+{pts})')
    for legacy, pts in [('heartBeatBAS',8),('heartBeatFAS',8),
                        ('heartBeatCCTV',6),('heartBeatTL',6)]:
        if is_offline(row.get(legacy)):
            s += pts; reasons.append(f'{legacy}_OFFLINE(+{pts})')

    # ── Voltage/current (spec §3) — skip when missing ────────────────────────
    bv_raw = row.get('battery_voltage')
    if not is_missing(bv_raw):
        bv = safe_float(bv_raw, -1)
        if 0 < bv < 11.0:
            s += 15; reasons.append(f'BATT_V_CRIT({bv:.1f}V)(+15)')
        elif 11.0 <= bv < 12.0:
            s += 8;  reasons.append(f'BATT_V_WARN({bv:.1f}V)(+8)')
    av_raw = row.get('ac_voltage')
    if not is_missing(av_raw):
        av = safe_float(av_raw, -1)
        if 0 < av < 180:
            s += 10; reasons.append(f'AC_V_LOW({av:.0f}V)(+10)')

    # ── Event log fault counts ───────────────────────────────────────────────
    fe = safe_int(row.get('log_fault_count', 0))
    if   fe >= 20: s += 20; reasons.append(f'FAULT_EVT={fe}(+20)')
    elif fe >= 10: s += 12; reasons.append(f'FAULT_EVT={fe}(+12)')
    elif fe >= 3:  s += 6;  reasons.append(f'FAULT_EVT={fe}(+6)')

    # ── SD card bad cards (parsed JSON) ──────────────────────────────────────
    bad_sd = (safe_int(row.get('hik_sd_bad_count',0))
            + safe_int(row.get('dahua_sd_bad_count',0))
            + safe_int(row.get('cpplus_sd_bad_count',0)))
    if bad_sd > 0:
        pts = min(bad_sd*5, 15); s += pts
        reasons.append(f'SD_BAD={bad_sd}(+{pts})')

    # ── Subsystem snapshots (server-scope) — SKIP IF MISSING ─────────────────
    # Only score when value is an explicit fault token. N/A/'' means subsystem
    # not applicable to this branch, not a fault.
    SNAP_RULES = [
        ('nvr_status_snapshot',         30,  'NVR_FAULT'),
        ('hdd_status_snapshot',         30,  'HDD_FAULT'),
        ('fas_status_snapshot',         20,  'FAS_FAULT'),
        ('ias_status_snapshot',         15,  'IAS_FAULT'),
        ('bas_status_snapshot',         12,  'BAS_FAULT'),
        ('acs_status_snapshot',         15,  'ACS_FAULT'),
        ('tls_status_snapshot',         10,  'TLS_FAULT'),
        ('gw_status_snapshot',          15,  'GW_FAULT'),
        ('cctv_status_snapshot',        10,  'CCTV_FAULT'),
        ('integrated_status_snapshot',  15,  'INT_FAULT'),
    ]
    for col, pts, label in SNAP_RULES:
        v = row.get(col)
        if is_missing(v): continue          # ← THE FIX
        if is_fault_val(v):
            s += pts; reasons.append(f'{label}(+{pts})')

    # ── BAS downtime (legacy, real numeric) ──────────────────────────────────
    bd_raw = row.get('BAS_Downtime_Minutes')
    if not is_missing(bd_raw):
        bd = safe_float(bd_raw, 0)
        if   bd >= 1440: s += 30; reasons.append(f'BAS_DT_{bd:.0f}m(+30)')
        elif bd >= 480:  s += 20; reasons.append(f'BAS_DT_{bd:.0f}m(+20)')
        elif bd >= 60:   s += 10; reasons.append(f'BAS_DT_{bd:.0f}m(+10)')

    # ── Data usage — use PARSED numeric, only fire ZERO_DATA when we know it's 0
    du = row.get('Total_Data_Usage_num')
    if du is not None and not (isinstance(du, float) and math.isnan(du)):
        if du == 0:
            s += 20; reasons.append('ZERO_DATA(+20)')

    # ── System metrics ───────────────────────────────────────────────────────
    for fld, raw in [
        ('cpu', row.get('cpu')), ('disk', row.get('disk')),
        ('memory', row.get('memory')), ('temperature', row.get('temperature')),
    ]:
        if is_missing(raw): continue
        v = safe_float(raw, 0)
        if fld == 'disk':
            if   v >= 90: s += 20; reasons.append(f'DISK_CRIT({v:.0f}%)')
            elif v >= 80: s += 12; reasons.append(f'DISK_HIGH({v:.0f}%)')
            elif v >= 75: s += 6;  reasons.append(f'DISK_WARN({v:.0f}%)')
        elif fld == 'cpu':
            if v >= 90: s += 10; reasons.append(f'CPU_HIGH({v:.0f}%)')
        elif fld == 'memory':
            if v >= 90: s += 10; reasons.append(f'MEM_HIGH({v:.0f}%)')
        elif fld == 'temperature':
            if   v >= 70: s += 12; reasons.append(f'TEMP_CRIT({v:.0f}°)')
            elif v >= 60: s += 6;  reasons.append(f'TEMP_WARN({v:.0f}°)')

    # ── Fault counts (real telemetry) ────────────────────────────────────────
    for cnt_key, label, weight_per, cap in [
        ('BASfaultCOUNT','BAS_F',3,12),
        ('FASfaultCOUNT','FAS_F',3,12),
        ('IASfaultCOUNT','IAS_F',3,12),
        ('camera_disconnect_count','CAM_DC',3,20),
        ('camera_tampered_count','CAM_TAMP',3,15),
        ('hdd_error_count','HDD_ERR',5,20),
    ]:
        v = row.get(cnt_key)
        if is_missing(v): continue
        c = safe_int(v, 0)
        if c > 0:
            pts = min(c*weight_per, cap); s += pts
            reasons.append(f'{label}={c}(+{pts})')

    sw = str(row.get('sw_state') or '').upper()
    if sw in ('FAILED','FAILED_UPDATE','ERROR'):
        s += 10; reasons.append('FW_FAIL(+10)')

    score = round(min(s, 100), 2)
    sev = ('CRITICAL' if score >= 70 else 'HIGH' if score >= 45
           else 'MEDIUM' if score >= 20 else 'HEALTHY')
    return score, sev, ' | '.join(reasons[:5]) or 'OK'


print(f'⚙️  Scoring {len(ts_device_df):,} daily snapshots ...')
ts_device_df[['ts_fault_score','ts_severity','ts_top_reasons']] = \
    ts_device_df.apply(lambda r: pd.Series(score_ts_row(r)), axis=1)

print(f'\n✅ Scored {len(ts_device_df):,} rows')

# Overall severity
sev_counts = ts_device_df['ts_severity'].value_counts()
print('\nSeverity distribution — ALL devices:')
for sev in ['CRITICAL','HIGH','MEDIUM','HEALTHY']:
    n = int(sev_counts.get(sev, 0))
    pct = 100*n//max(len(ts_device_df),1)
    bar = '█' * (pct//3)
    print(f'  {sev:<10} {n:>8}  ({pct}%)  {bar}')

# Severity for REAL devices only — what matters
if 'is_real_device' in ts_device_df.columns:
    real_df = ts_device_df[ts_device_df['is_real_device']]
    real_sev = real_df['ts_severity'].value_counts()
    print(f'\nSeverity distribution — REAL devices only ({real_df["device_id"].nunique()} devices):')
    for sev in ['CRITICAL','HIGH','MEDIUM','HEALTHY']:
        n = int(real_sev.get(sev, 0))
        pct = 100*n//max(len(real_df),1)
        bar = '█' * (pct//3)
        print(f'  {sev:<10} {n:>8}  ({pct}%)  {bar}')

# Per-device avg score distribution
print(f'\nPer-device avg-score histogram (real devices only):')
if 'is_real_device' in ts_device_df.columns:
    per_dev = (ts_device_df[ts_device_df['is_real_device']]
               .groupby('device_id')['ts_fault_score'].mean())
    buckets = [0]*5
    labels = ['0-19','20-39','40-59','60-79','80-100']
    for s in per_dev:
        if s >= 80: buckets[4] += 1
        elif s >= 60: buckets[3] += 1
        elif s >= 40: buckets[2] += 1
        elif s >= 20: buckets[1] += 1
        else: buckets[0] += 1
    for lab, n in zip(labels, buckets):
        bar = '█' * (n//2)
        print(f'  avg {lab:<8}: {n:>4} devices  {bar}')


⚙️  Scoring 67,344 daily snapshots ...



✅ Scored 67,344 rows

Severity distribution — ALL devices:
  CRITICAL       1337  (1%)  
  HIGH           4597  (6%)  ██
  MEDIUM        10088  (14%)  ████
  HEALTHY       51322  (76%)  █████████████████████████



Severity distribution — REAL devices only (128 devices):
  CRITICAL       1051  (2%)  
  HIGH           4296  (9%)  ███
  MEDIUM         6253  (13%)  ████
  HEALTHY       35248  (75%)  █████████████████████████

Per-device avg-score histogram (real devices only):


  avg 0-19    :   99 devices  █████████████████████████████████████████████████
  avg 20-39   :   13 devices  ██████
  avg 40-59   :   11 devices  █████
  avg 60-79   :    2 devices  █
  avg 80-100  :    3 devices  █


---
## Cell 19 — Export Time-Series ML JSONL + Excel + Trend Summary

Produces:
- `ml_training_timeseries.jsonl` — historical training examples (10–60k rows)
- `ts_summary.xlsx` — 3 sheets: daily snapshots, per-device trend, severity by bank+date


In [19]:
import json
import os
import math
import re
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook

TS_JSONL_PATH    = 'ml_training_timeseries.jsonl'
TS_XLSX_PATH     = f'ts_summary_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'
TS_CSV_FULL_PATH = f'ts_daily_snapshots_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
COMBINED_JSONL   = 'ml_training_combined.jsonl'
DASHBOARD_JSON   = f'dashboard_data_{datetime.now().strftime("%Y%m%d_%H%M")}.json'

def _col(k):
    return _TS_COLNAME.get(k, k) if '_TS_COLNAME' in dir() else k

# ── Trim columns ──────────────────────────────────────────────────────────────
print('🧹 Trimming zero-coverage columns ...')
keep_cols, drop_cols = [], []
ALWAYS_KEEP = {'device_id','date','device_name','bank_name','ho_name','nbg_name',
               'lho_name','zo_name','ro_name','co_name','rbo_name','branch_name',
               'branch_code','full_path','hierarchy_depth','nvr_brand',
               'is_real_device','ts_fault_score','ts_severity','ts_top_reasons'}
for c in ts_device_df.columns:
    if c in ALWAYS_KEEP:
        keep_cols.append(c); continue
    col = ts_device_df[c]
    has = (col.astype(str).str.strip().str.lower()
           .isin(_MISSING_STR | {''}).eq(False).sum()
           if col.dtype == object else col.notna().sum())
    (drop_cols if has == 0 else keep_cols).append(c)
ts_trim = ts_device_df[keep_cols].copy()
print(f'   Kept {len(keep_cols)} cols, dropped {len(drop_cols)} zero-coverage cols')

HEAVY_PATTERNS = ('CameraRecInfo','cameraInfo','rec_info_list','HDDInfo','deviceAllInfo')
heavy = [c for c in ts_trim.columns if any(p in c for p in HEAVY_PATTERNS)]
ts_trim = ts_trim.drop(columns=heavy, errors='ignore')
print(f'   Dropped {len(heavy)} heavy raw-JSON columns')
print(f'   Final shape: {ts_trim.shape}')

print('🧵 Vectorized stringification ...')
for c in ts_trim.select_dtypes(include=['object']).columns:
    s = ts_trim[c].astype(str)
    mask = s.str.len() > 32000
    if mask.any():
        s = s.where(~mask, s.str.slice(0, 32000))
    ts_trim[c] = s.replace({'None':'','nan':'','NaN':''})

# ── Column → feature-group mapping (broad classifier) ────────────────────────
HIERARCHY_COLS = {
    'device_id','device_name','bank_name','ho_name','nbg_name','lho_name',
    'zo_name','ro_name','co_name','rbo_name','branch_name','branch_code','branch_id',
    'full_path','hierarchy_depth','nvr_brand','is_real_device','date','device_type',
    'deviceType','customer_name','ts_branch','ts_nbg','ts_zo','ts_device_type',
    'originatorId','originatorName','originatorType','dev_id',
}
SCORING_COLS = {'ts_fault_score','ts_severity','ts_top_reasons'}

def _h(s, *needles):
    sl = s.lower()
    return any(n in sl for n in needles)

def _classify(c):
    cl = c.lower()
    if c in HIERARCHY_COLS: return 'hierarchy'
    if c in SCORING_COLS:   return 'scoring'
    if c.endswith('_status_snapshot'): return 'status_snapshot'
    if c.startswith('statusbox_') or c.startswith('ts_statusbox_') or c == 'system_status':
        return 'statusbox'
    if _h(cl, 'sd_card', 'sd_cam', 'sd_bad', 'sd_info', 'sdrec'):
        return 'sd_card'
    # CCTV / NVR / HDD / cameras — broad catch BEFORE subsystems
    if _h(cl, 'camera', 'nvr', 'hdd', 'hik', 'dahua', 'cpplus', 'cp_plus',
            'recording', 'channel', 'chnl', 'videoinfo', 'videodetails',
            'video_details', 'dvr', 'cctv', 'cdisc', 'chdis', 'all_ch_disconnect',
            'ch_tamper'):
        return 'cctv'
    # BAS subsystem (includes gateway)
    if (cl.startswith('bas') or c.startswith('BAS') or
            _h(cl, 'bacs', 'gateway') or c.startswith('GW_') or
            c.startswith('Gateway') or cl.startswith('gw_')):
        return 'bas'
    if (cl.startswith('fas') or c.startswith('FAS') or
            _h(cl, 'fire_alarm', 'firealarm') or c in ('fasf', 'score_fas')):
        return 'fas'
    if (cl.startswith('ias') or c.startswith('IAS') or
            _h(cl, 'intrusion', 'instrusion', 'integrated_alarm') or
            c == 'iasf' or cl.startswith('zone')):
        return 'ias'
    if (cl.startswith('acs') or c.startswith('ACS') or
            _h(cl, 'access_control', 'accesscontrol', 'autodialer') or
            c in ('acst', 'acd')):
        return 'acs'
    if (cl.startswith('tls') or c.startswith('TLS') or
            _h(cl, 'time_lock', 'timelock', 'antidismantle') or
            (cl.startswith('tl') and (c.endswith(('TS','Ts','Time','Duration')) or
                                      c in ('tld','tlt')))):
        return 'tls'
    if _h(cl, 'texecom') or c.startswith('tex_'): return 'texecom'
    if _h(cl, 'heartbeat', 'lasthour', 'offtime', 'uptime',
            'heartbeatstop', 'heartbeatcount', 'heartbeatts'):
        return 'heartbeat'
    if ((_h(cl, 'battery', 'batt_', 'bat_', 'ac_volt', 'ac_result',
             'ac_status', 'ac_inactive', 'ac_fault', 'acinactive', 'acfault',
             'current_status', 'system_current', 'power_', 'capacity',
             'cur_result', 'ups_status') or cl.startswith('ac')) and
            not _h(cl, 'firmware', 'hardware', 'sw_', 'access', 'acs')):
        return 'power'
    if (cl in {'lat','lon','lat1','lon1','latitude','longitude','arrlat','arrlon',
               'totallat','totallon','geostatus','area','lat_lon_default'} or
            c in {'restrictedAreaStatus','loadingAreaStatus',
                  'unloadingAreaStatus','mineSiteAreaStatus'}):
        return 'gps'
    if (_h(cl, 'downtime', 'uptime_') or
            c in {'BAS_Uptime_%','BAS_Hang_Events','BAS_Expected_Heartbeats',
                  'BAS_Received_Heartbeats'} or
            c.endswith('_minutes') or c.endswith('_Minutes')):
        return 'downtime'
    if (cl.startswith('log_') or
            _h(cl, 'alarm', 'error', 'event', 'fault', 'ticket', 'watchdog',
                 'alert', 'message', '_ecd_') or
            c in {'Device_Issue', 'mail', 'ai_event'}):
        return 'events'
    if _h(cl, 'data_usage', 'net_recv', 'net_sent', 'tailscale', 'dnshostname',
            'ipaddress', 'signal_strength', 'rssi', 'operator', 'sim_status',
            'network', 'expires_in', 'access_token', 'token_type', 'clientid',
            'api_domain', 'topic', 'scope', 'imei', 'ipcom', 'comip',
            'frequency', 'heartbeat_7d', 'total_data', 'usage', 'host',
            'total_active', 'total_capacity', 'total_heartbeats'):
        return 'network'
    if cl in {'cpu','disk','memory','temperature','humidity','uptimetotal',
              'cavlidata_ontime','sw_state','sw_version','firmwareversion',
              'hardwareversion','processor','manufacturer','model',
              'serialnumber','mfdate','version','type2','sensor_id','mode',
              'panel_mode','msiren_sts','siren_sts','tamper_sts',
              'notification_sts','ctemp','active','sys','svk','pri','emcp',
              'totalactivedevices','doorlockstatus','doorstatus',
              'magneticstatus','totalactive','totalcapacity','totalheartbeats',
              'noofhddslots','freespace','count_hdd','count_ch','device_status'}:
        return 'system'
    if c.startswith('target_sw_'): return 'system'
    if cl in {'rock','rockai','notificationsettings','notificationusers',
              'contacts','calculatets','calculatedmonth','composite_key',
              'composite_keys','sum','sum_before_ts','dexter_date','dexter_time',
              'synctimedate','cpdatetime','hiksynctimedate','dahuasynctimedate',
              'localtime','timemode','timezone','date','time','ts_date',
              'ts_time','ts_alarm','ts_attribute','ts_id','ts_status',
              'status_device','healthystatus','dailyfuelconsumption',
              'hourlyfuelconsumption','payload','timestamp','fault_ts',
              'panel_info','slot','cam_total','cam_dc_count_snapshot',
              'cam_offline_snapshot','cam_online_snapshot',
              'cam_tamper_count_snapshot'} or c.startswith('rpi_'):
        return 'diagnostic'
    return 'other'

GROUP_OF = {c: _classify(c) for c in ts_trim.columns}
GROUP_BUCKETS = {}
for c, g in GROUP_OF.items():
    GROUP_BUCKETS.setdefault(g, []).append(c)
print('   Feature groups:', {g: len(cs) for g, cs in GROUP_BUCKETS.items()})

# ── JSONL — rich features per row ─────────────────────────────────────────────
print(f'\n📝 Writing JSONL → {TS_JSONL_PATH} ...')

STATUSBOX_KEYS = [
    'statusbox_system_on','statusbox_system_healthy','statusbox_mains_on',
    'statusbox_battery_reverse','statusbox_battery_low','statusbox_sos_status',
    'statusbox_network','statusbox_no_of_connected_device',
]
HEARTBEAT_KEYS = [
    'heartbeat_BAS','heartbeat_FAS','heartbeat_CCTV','heartbeat_IBAS',
    'heartbeat_access_control','heartbeat_time_lock',
    'heartBeatBAS','heartBeatFAS','heartBeatCCTV','heartBeatTL',
]
SNAPSHOT_STATUS_KEYS = [
    'nvr_status_snapshot','hdd_status_snapshot','fas_status_snapshot',
    'ias_status_snapshot','bas_status_snapshot','acs_status_snapshot',
    'tls_status_snapshot','gw_status_snapshot','cctv_status_snapshot',
    'integrated_status_snapshot',
]
NUMERIC_KEYS = [
    ('battery_voltage','batt_v','%.1f'),('ac_voltage','ac_v','%.0f'),
    ('system_current','sys_i','%.2f'),
    ('lat','lat','%.4f'),('lon','lon','%.4f'),
    ('log_event_count','evt_all','%.0f'),('log_fault_count','evt_fault','%.0f'),
    ('hik_sd_bad_count','hik_sd_bad','%.0f'),
    ('dahua_sd_bad_count','dah_sd_bad','%.0f'),
    ('cpplus_sd_bad_count','cpp_sd_bad','%.0f'),
    ('Total_Data_Usage_num','data_mb','%.0f'),
    ('BAS_Downtime_Minutes','bas_dt_min','%.0f'),
    ('BAS_Uptime_Minutes','bas_up_min','%.0f'),
    ('cavlidata_ontime','cavli_ontime','%.0f'),
    ('cpu','cpu','%.0f'),('disk','disk','%.0f'),
    ('memory','mem','%.0f'),('temperature','temp','%.0f'),
    ('BASfaultCOUNT','bas_f_cnt','%.0f'),('FASfaultCOUNT','fas_f_cnt','%.0f'),
    ('IASfaultCOUNT','ias_f_cnt','%.0f'),
    ('BASinactiveCOUNT','bas_inact','%.0f'),('FASinactiveCOUNT','fas_inact','%.0f'),
    ('camera_disconnect_count','cam_dc_cnt','%.0f'),
    ('camera_tampered_count','cam_tp_cnt','%.0f'),
    ('hdd_error_count','hdd_err_cnt','%.0f'),
    ('alarmCount','alarm_cnt','%.0f'),('errorCount','err_cnt','%.0f'),
    ('uptimeTotal','uptime_total','%.0f'),
]

cols_list = list(ts_device_df.columns)
col_idx   = {c: i for i, c in enumerate(cols_list)}

# columns to actually emit in `features` (skip heavy raw JSON, but keep
# everything that survived ts_trim — same data the CSV carries)
FEATURE_COLS = [c for c in ts_trim.columns if c in col_idx]
FEATURE_COL_GROUP = {c: GROUP_OF.get(c, 'other') for c in FEATURE_COLS}

def _g(row, c, d=''):
    i = col_idx.get(c)
    if i is None: return d
    v = row[i]
    return d if v is None else v

def _is_missing_scalar(v):
    if v is None: return True
    if isinstance(v, float):
        try:
            if math.isnan(v) or math.isinf(v): return True
        except Exception:
            pass
        return False
    if isinstance(v, str):
        return v.strip().lower() in (_MISSING_STR | {''})
    return False

def _coerce_value(v):
    """Make a value JSON-serializable AND useful: ints/floats stay numeric,
    booleans stay boolean, JSON-looking strings are parsed if cheap, the rest
    stays string."""
    if v is None: return None
    if isinstance(v, (bool, int)): return v
    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v): return None
        if v.is_integer(): return int(v)
        return round(v, 4)
    s = str(v).strip()
    if not s or s.lower() in (_MISSING_STR | {''}):
        return None
    if s.lstrip('-').replace('.','',1).isdigit():
        try:
            n = float(s)
            if math.isnan(n) or math.isinf(n): return None
            return int(n) if n.is_integer() else round(n, 4)
        except Exception:
            return s
    if (s.startswith('{') and s.endswith('}')) or (s.startswith('[') and s.endswith(']')):
        try:
            return json.loads(s)
        except Exception:
            return s
    if s.lower() in ('true','false'):
        return s.lower() == 'true'
    return s

written = skipped = 0
with open(TS_JSONL_PATH, 'w', encoding='utf-8') as f:
    for row in ts_device_df.itertuples(index=False, name=None):
        # ── slim training prompt (unchanged) ─────────────────────────────────
        parts = []
        bn = _g(row,'bank_name')
        if bn: parts.append(f'bank:{bn}')
        ng = _g(row,'nbg_name')
        if ng: parts.append(f'nbg:{ng}')
        zn = _g(row,'zo_name')
        if zn: parts.append(f'zo:{zn}')
        br = _g(row,'branch_name')
        if br: parts.append(f'branch:{br}')
        parts.append(f'date:{_g(row,"date")}')
        nb = _g(row,'nvr_brand')
        if nb: parts.append(f'nvr_brand:{nb}')

        for k in SNAPSHOT_STATUS_KEYS:
            v = str(_g(row,k) or '').strip()
            if v and v.lower() not in _MISSING_STR:
                parts.append(f'{k.replace("_snapshot","")}:{v}')

        for k in STATUSBOX_KEYS:
            v = str(_g(row, _col(k)) or '').strip()
            if v and v.lower() not in _MISSING_STR:
                parts.append(f'{k}:{v}')

        for k in HEARTBEAT_KEYS:
            v = str(_g(row, _col(k)) or '').strip().lower()
            if v in ('online','offline'):
                parts.append(f'{k}:{v}')

        sw = str(_g(row, _col('sw_state')) or '').strip()
        if sw and sw.lower() not in _MISSING_STR:
            parts.append(f'sw_state:{sw}')

        for src, short, fmt in NUMERIC_KEYS:
            col_name = src if src in col_idx else _col(src)
            v = _g(row, col_name, None)
            if v is None: continue
            try: vf = float(v)
            except Exception: continue
            if math.isnan(vf) or math.isinf(vf): continue
            if vf == 0: continue
            parts.append(f'{short}:{fmt % vf}')

        hd = _g(row,'hierarchy_depth')
        if hd: parts.append(f'hier_depth:{hd}')

        if len(parts) < 3:
            skipped += 1
            continue

        # ── rich features (NEW — same data the CSV row carries) ──────────────
        features_flat = {}
        features_grouped = {}
        for c in FEATURE_COLS:
            raw = _g(row, c, None)
            if _is_missing_scalar(raw): continue
            val = _coerce_value(raw)
            if val is None: continue
            features_flat[c] = val
            g = FEATURE_COL_GROUP[c]
            features_grouped.setdefault(g, {})[c] = val

        f.write(json.dumps({
            'input':  ' '.join(parts),
            'output': (
                f'fault_class:{_g(row,"ts_severity")} '
                f'fault_score:{float(_g(row,"ts_fault_score") or 0):.0f} '
                f'reasons:{_g(row,"ts_top_reasons")}'
            ),
            'features':          features_flat,
            'features_grouped':  features_grouped,
            '_meta': {
                'device_id':       _g(row,'device_id'),
                'date':            _g(row,'date'),
                'bank':            _g(row,'bank_name'),
                'branch':          _g(row,'branch_name'),
                'severity':        _g(row,'ts_severity'),
                'score':           _g(row,'ts_fault_score'),
                'is_real_device':  bool(_g(row,'is_real_device', True)),
                'feature_count':   len(features_flat),
            },
        }, ensure_ascii=False) + '\n')
        written += 1

print(f'✅ TS JSONL: {written:,} examples ({skipped:,} skipped)')

# ── Combined JSONL ────────────────────────────────────────────────────────────
combined = 0
with open(COMBINED_JSONL, 'w', encoding='utf-8') as out:
    for src in ['ml_training_v11.jsonl', TS_JSONL_PATH]:
        if os.path.exists(src):
            with open(src,'r',encoding='utf-8') as inp:
                for line in inp:
                    line = line.strip()
                    if line:
                        out.write(line + '\n')
                        combined += 1
print(f'✅ Combined JSONL: {combined:,} total examples')

# ── CSV (full) ────────────────────────────────────────────────────────────────
print(f'\n💾 Writing daily snapshots → {TS_CSV_FULL_PATH} ...')
ts_trim.to_csv(TS_CSV_FULL_PATH, index=False, encoding='utf-8-sig')
print(f'   ✅ {os.path.getsize(TS_CSV_FULL_PATH)//1024} KB')

# ── XLSX summary (filtered to REAL devices for risk views) ───────────────────
print(f'\n📊 Building summary xlsx → {TS_XLSX_PATH} ...')

def _sum_num(x):  return round(pd.to_numeric(x, errors='coerce').fillna(0).sum(), 1)
def _zero_num(x): return (pd.to_numeric(x, errors='coerce').fillna(-1) == 0).sum()
def _isum(x):     return int(pd.to_numeric(x, errors='coerce').fillna(0).sum())
def _mean_num(x): return round(pd.to_numeric(x, errors='coerce').mean(), 1)

real_df = (ts_device_df[ts_device_df['is_real_device']]
           if 'is_real_device' in ts_device_df.columns else ts_device_df)
print(f'   Real-device rows: {len(real_df):,} (of {len(ts_device_df):,} total)')

group_keys = [c for c in
              ['device_id','device_name','bank_name','nbg_name','zo_name','branch_name']
              if c in real_df.columns]

agg_dict = {
    'days_with_data':  ('ts_fault_score','count'),
    'avg_fault_score': ('ts_fault_score','mean'),
    'max_fault_score': ('ts_fault_score','max'),
    'days_critical':   ('ts_severity', lambda x: (x=='CRITICAL').sum()),
    'days_high':       ('ts_severity', lambda x: (x=='HIGH').sum()),
    'days_healthy':    ('ts_severity', lambda x: (x=='HEALTHY').sum()),
}
def _add(col, name, fn):
    if col in real_df.columns:
        agg_dict[name] = (col, fn)
_add('BAS_Downtime_Minutes','total_bas_dt_hrs', lambda x: round(_sum_num(x)/60,1))
_add('Total_Data_Usage_num','total_data_mb', _sum_num)
_add('log_fault_count','total_fault_evts', _isum)
_add('cpu','avg_cpu',_mean_num)
_add('disk','avg_disk',_mean_num)
_add('temperature','avg_temp',_mean_num)
_add('BASfaultCOUNT','total_bas_faults',_isum)
_add('FASfaultCOUNT','total_fas_faults',_isum)
_add('camera_disconnect_count','total_cam_dc',_isum)
_add('hdd_error_count','total_hdd_errors',_isum)

try:
    trend_df = (real_df.groupby(group_keys, dropna=False)
                .agg(**agg_dict).reset_index()
                .sort_values('avg_fault_score', ascending=False))
    trend_df['avg_fault_score'] = trend_df['avg_fault_score'].round(1)
except Exception as e:
    print(f'   ⚠ Trend agg failed: {e}')
    trend_df = real_df.groupby(group_keys, dropna=False)['ts_fault_score'] \
                      .agg(['count','mean','max']).reset_index()

heat_agg = {
    'devices':   ('device_id','nunique'),
    'avg_score': ('ts_fault_score','mean'),
    'critical':  ('ts_severity', lambda x: (x=='CRITICAL').sum()),
    'high':      ('ts_severity', lambda x: (x=='HIGH').sum()),
}
if 'log_fault_count' in real_df.columns:
    heat_agg['fault_evts'] = ('log_fault_count', _isum)

try:
    heat_df = (real_df.groupby(['bank_name','date'], dropna=False)
               .agg(**heat_agg).reset_index())
    heat_df['avg_score'] = heat_df['avg_score'].round(1)
except Exception as e:
    print(f'   ⚠ Heat agg failed: {e}')
    heat_df = pd.DataFrame()

top_risk = (real_df[real_df['ts_severity'].isin(['CRITICAL','HIGH'])]
            [['date','device_name','bank_name','branch_name',
              'ts_fault_score','ts_severity','ts_top_reasons']]
            .sort_values('ts_fault_score', ascending=False)
            .head(500))

HDR=PatternFill('solid',fgColor='1B3A5C'); HF=Font(bold=True,color='FFFFFF')
HA=Alignment(horizontal='center',vertical='center',wrap_text=True)
FILLS={'CRITICAL':PatternFill('solid',fgColor='FFCCCC'),
       'HIGH':PatternFill('solid',fgColor='FFE5CC'),
       'MEDIUM':PatternFill('solid',fgColor='FFFACC'),
       'HEALTHY':PatternFill('solid',fgColor='CCFFCC')}
def style_ws(ws, sev_col=None):
    for c in ws[1]: c.fill=HDR; c.font=HF; c.alignment=HA
    ws.row_dimensions[1].height=28
    ws.freeze_panes='A2'
    for col in ws.columns:
        w=max((len(str(c.value or '')) for c in col),default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width=min(w+3,42)
    if sev_col:
        sc=next((c.column for c in ws[1] if str(c.value)==sev_col),None)
        if sc:
            for r in ws.iter_rows(min_row=2,min_col=sc,max_col=sc):
                for cell in r: cell.fill=FILLS.get(str(cell.value),PatternFill())

try:
    with pd.ExcelWriter(TS_XLSX_PATH, engine='openpyxl') as w:
        trend_df.to_excel(w, sheet_name='Device Trend', index=False)
        if not heat_df.empty:
            heat_df.to_excel(w, sheet_name='Bank×Date Heat', index=False)
        top_risk.to_excel(w, sheet_name='Top Risk Days', index=False)
    wb = load_workbook(TS_XLSX_PATH)
    style_ws(wb['Device Trend'])
    if 'Bank×Date Heat' in wb.sheetnames: style_ws(wb['Bank×Date Heat'])
    style_ws(wb['Top Risk Days'], sev_col='ts_severity')
    wb.save(TS_XLSX_PATH)
    print(f'   ✅ {TS_XLSX_PATH} ({os.path.getsize(TS_XLSX_PATH)//1024} KB)')
except PermissionError:
    print('   ❌ Close the open ts_summary_*.xlsx and re-run Cell 19.')
except Exception as e:
    print(f'   ❌ Excel export failed: {type(e).__name__}: {e}')
    import traceback; traceback.print_exc()

# ── Structured dashboard JSON (v2 — hierarchy tree + flat views) ──────────────
print(f'\n📦 Writing dashboard JSON → {DASHBOARD_JSON} ...')

def _records(df, limit=None):
    if df is None or df.empty: return []
    out = df if limit is None else df.head(limit)
    cleaned = []
    for rec in out.to_dict(orient='records'):
        clean = {}
        for k, v in rec.items():
            if isinstance(v, float):
                if math.isnan(v) or math.isinf(v): continue
                clean[k] = round(v, 4) if not float(v).is_integer() else int(v)
            elif v is None or (isinstance(v, str) and v.strip().lower() in (_MISSING_STR | {''})):
                continue
            else:
                clean[k] = v
        cleaned.append(clean)
    return cleaned

def _ts_agg(devs):
    if not devs: return {'device_count': 0}
    scores = [d['score_avg'] for d in devs if d.get('score_avg') is not None]
    return {
        'device_count':        len(devs),
        'avg_score':           round(sum(scores) / len(scores), 1) if scores else None,
        'deteriorating':       sum(1 for d in devs if str(d.get('trend_direction', '')) == 'DETERIORATING'),
        'stable':              sum(1 for d in devs if str(d.get('trend_direction', '')) == 'STABLE'),
        'recovering':          sum(1 for d in devs if str(d.get('trend_direction', '')) == 'RECOVERING'),
        'total_critical_days': sum(int(d.get('days_critical', 0) or 0) for d in devs),
    }

# Hierarchy tree from per-device trend summaries: bank→NBG→zone→branch→devices
print('   Building timeseries hierarchy tree ...')
ts_hier_tree = {}
for dev in _records(trend_df):
    bank   = str(dev.get('bank_name',   'Unknown') or 'Unknown').strip()
    nbg    = str(dev.get('nbg_name',    'Unknown') or 'Unknown').strip()
    zo     = str(dev.get('zo_name',     'Unknown') or 'Unknown').strip()
    branch = str(dev.get('branch_name', 'Unknown') or 'Unknown').strip()
    b  = ts_hier_tree.setdefault(bank,   {'_summary': {}, 'nbg': {}})
    n  = b['nbg'].setdefault(nbg,        {'_summary': {}, 'zones': {}})
    z  = n['zones'].setdefault(zo,       {'_summary': {}, 'branches': {}})
    br = z['branches'].setdefault(branch, {'_summary': {}, 'devices': []})
    br['devices'].append(dev)

for bank, bdata in ts_hier_tree.items():
    bk_all = []
    for nbg, ndata in bdata['nbg'].items():
        nb_all = []
        for zo, zdata in ndata['zones'].items():
            zo_all = []
            for branch, brdata in zdata['branches'].items():
                brdata['_summary'] = _ts_agg(brdata['devices']); zo_all += brdata['devices']
            zdata['_summary'] = _ts_agg(zo_all); nb_all += zo_all
        ndata['_summary'] = _ts_agg(nb_all); bk_all += nb_all
    bdata['_summary'] = _ts_agg(bk_all)

total_zos = sum(len(n['zones']) for b in ts_hier_tree.values() for n in b['nbg'].values())
print(f'   Timeseries tree: {len(ts_hier_tree)} banks, {total_zos} zones')

sev_counts_all  = ts_device_df['ts_severity'].value_counts().to_dict() if 'ts_severity' in ts_device_df.columns else {}
sev_counts_real = real_df['ts_severity'].value_counts().to_dict() if 'ts_severity' in real_df.columns else {}

dashboard = {
    'schema_version': 2,
    'generated_at':   datetime.now().isoformat(timespec='seconds'),
    'window': {
        'start_date': str(ts_device_df['date'].min()) if 'date' in ts_device_df.columns and not ts_device_df.empty else None,
        'end_date':   str(ts_device_df['date'].max()) if 'date' in ts_device_df.columns and not ts_device_df.empty else None,
    },
    'totals': {
        'total_device_days': int(len(ts_device_df)),
        'real_device_days':  int(len(real_df)),
        'total_devices':     int(ts_device_df['device_id'].nunique()),
        'real_devices':      int(real_df['device_id'].nunique()) if not real_df.empty else 0,
        'feature_columns':   int(len(ts_trim.columns)),
        'total_banks':       int(ts_device_df['bank_name'].nunique() if 'bank_name' in ts_device_df.columns else 0),
        'total_branches':    int(ts_device_df['branch_name'].nunique() if 'branch_name' in ts_device_df.columns else 0),
    },
    'severity_distribution': {
        'all_devices':  {str(k): int(v) for k, v in sev_counts_all.items()},
        'real_devices': {str(k): int(v) for k, v in sev_counts_real.items()},
    },
    'feature_groups': {g: sorted(cs) for g, cs in GROUP_BUCKETS.items()},
    'hierarchy_tree': ts_hier_tree,
    'flat_views': {
        'device_trend':   _records(trend_df),
        'bank_date_heat': _records(heat_df) if not heat_df.empty else [],
        'top_risk_days':  _records(top_risk, limit=500),
    },
}
try:
    with open(DASHBOARD_JSON, 'w', encoding='utf-8') as f:
        json.dump(dashboard, f, ensure_ascii=False, indent=2, default=str)
    print(f'   ✅ {DASHBOARD_JSON} ({os.path.getsize(DASHBOARD_JSON)//1024} KB)')
except Exception as e:
    print(f'   ❌ Dashboard JSON failed: {type(e).__name__}: {e}')
    import traceback; traceback.print_exc()

print(f'\n📊 FINAL SUMMARY:')
print(f'   Daily snapshots CSV : {TS_CSV_FULL_PATH}  (all devices, {len(ts_trim.columns)} cols)')
print(f'   Summary XLSX        : {TS_XLSX_PATH}  (REAL devices only for risk views)')
print(f'   Dashboard JSON      : {DASHBOARD_JSON}  (hierarchy tree: bank→NBG→zone→branch + flat views)')
print(f'   TS JSONL            : {TS_JSONL_PATH}  ({written:,} examples, rich features per row)')
print(f'   Combined JSONL      : {COMBINED_JSONL}  ({combined:,} examples)')
print(f'   Real devices        : {real_df["device_id"].nunique() if not real_df.empty else 0}')
print(f'   Total devices       : {ts_device_df["device_id"].nunique()}')


🧹 Trimming zero-coverage columns ...


   Kept 646 cols, dropped 0 zero-coverage cols
   Dropped 9 heavy raw-JSON columns
   Final shape: (67344, 637)
🧵 Vectorized stringification ...


   Feature groups: {'hierarchy': 29, 'status_snapshot': 11, 'cctv': 241, 'power': 21, 'diagnostic': 37, 'system': 40, 'events': 22, 'ias': 30, 'statusbox': 9, 'heartbeat': 25, 'acs': 32, 'tls': 29, 'sd_card': 11, 'texecom': 6, 'gps': 17, 'network': 24, 'bas': 25, 'fas': 24, 'downtime': 1, 'scoring': 3}

📝 Writing JSONL → ml_training_timeseries.jsonl ...


✅ TS JSONL: 61,978 examples (5,366 skipped)


✅ Combined JSONL: 62,149 total examples

💾 Writing daily snapshots → ts_daily_snapshots_20260529_1559.csv ...


   ✅ 465863 KB

📊 Building summary xlsx → ts_summary_20260529_1559.xlsx ...


   Real-device rows: 46,848 (of 67,344 total)


   ✅ ts_summary_20260529_1559.xlsx (203 KB)

📦 Writing dashboard JSON → dashboard_data_20260529_1559.json ...
   Building timeseries hierarchy tree ...
   Timeseries tree: 19 banks, 35 zones
   ✅ dashboard_data_20260529_1559.json (1879 KB)

📊 FINAL SUMMARY:
   Daily snapshots CSV : ts_daily_snapshots_20260529_1559.csv  (all devices, 637 cols)
   Summary XLSX        : ts_summary_20260529_1559.xlsx  (REAL devices only for risk views)
   Dashboard JSON      : dashboard_data_20260529_1559.json  (hierarchy tree: bank→NBG→zone→branch + flat views)
   TS JSONL            : ml_training_timeseries.jsonl  (61,978 examples, rich features per row)
   Combined JSONL      : ml_training_combined.jsonl  (62,149 examples)
   Real devices        : 128
   Total devices       : 184


---
## Cell 20 — Fault Trend Analysis (Top Deteriorating Devices)

Shows which devices got **worse over time** (score trending up) vs
which **recovered** (score trending down). Useful for proactive maintenance.


In [20]:
import numpy as np

# ── Linear trend slope per device (fault score over time) ─────────────────────
print('📈 Computing fault trend (slope) per device ...')

trend_rows = []
for dev_id in ts_device_df['device_id'].unique():
    sub = ts_device_df[ts_device_df['device_id']==dev_id].copy()
    sub = sub.sort_values('date').reset_index(drop=True)

    if len(sub) < 7: continue  # need at least a week of data

    scores     = sub['ts_fault_score'].fillna(0).values
    x          = np.arange(len(scores))

    try:
        slope, intercept = np.polyfit(x, scores, 1)
    except:
        slope = 0.0

    h = hier_lookup.get(dev_id, {})
    trend_rows.append({
        'device_id':       dev_id,
        'device_name':     h.get('device_name', ''),
        'bank_name':       h.get('bank_name',''),
        'nbg_name':        h.get('nbg_name',''),
        'zo_name':         h.get('zo_name',''),
        'branch_name':     h.get('branch_name',''),
        'full_path':       h.get('full_path',''),
        'score_start':     round(float(scores[0]), 1),
        'score_end':       round(float(scores[-1]), 1),
        'score_avg':       round(float(scores.mean()), 1),
        'score_max':       round(float(scores.max()), 1),
        'trend_slope':     round(float(slope), 3),
        'trend_direction': 'DETERIORATING' if slope > 0.1
                           else ('RECOVERING' if slope < -0.1 else 'STABLE'),
        'days_critical':   int((sub['ts_severity']=='CRITICAL').sum()),
        'days_healthy':    int((sub['ts_severity']=='HEALTHY').sum()),
        'data_points':     len(scores),
    })

trend_analysis_df = pd.DataFrame(trend_rows).sort_values('trend_slope', ascending=False)

print(f'\n✅ Trend analysis: {len(trend_analysis_df)} devices')
direction_counts = trend_analysis_df['trend_direction'].value_counts()
print('\nTrend direction breakdown:')
for d, n in direction_counts.items():
    print(f'   {d:<15} {n}')

print('\n🔴 TOP 10 DETERIORATING DEVICES (worst trend slope):')
print(trend_analysis_df[
    trend_analysis_df['trend_direction']=='DETERIORATING'
][['device_name','bank_name','zo_name','score_start','score_end',
   'score_avg','trend_slope','days_critical']]
.head(10).to_string(index=False))

print('\n🟢 TOP 10 RECOVERING DEVICES (best improvement):')
print(trend_analysis_df[
    trend_analysis_df['trend_direction']=='RECOVERING'
].sort_values('trend_slope')[['device_name','bank_name','zo_name',
   'score_start','score_end','score_avg','trend_slope']]
.head(10).to_string(index=False))

# Append trend analysis to the TS Excel
with pd.ExcelWriter(TS_XLSX_PATH, engine='openpyxl', mode='a') as w:
    trend_analysis_df.to_excel(w, sheet_name='Trend Analysis', index=False)

wb = load_workbook(TS_XLSX_PATH)
style_ws(wb['Trend Analysis'])
wb.save(TS_XLSX_PATH)
print(f'\n✅ Trend analysis sheet added to {TS_XLSX_PATH}')


📈 Computing fault trend (slope) per device ...



✅ Trend analysis: 184 devices

Trend direction breakdown:
   STABLE          178
   DETERIORATING   6

🔴 TOP 10 DETERIORATING DEVICES (worst trend slope):
   device_name     bank_name    zo_name  score_start  score_end  score_avg  trend_slope  days_critical
    BOI-MISROD BANK OF INDIA ZO KHANDWA          0.0       45.0       25.6        0.170              0
BOI-SEONIMALWA BANK OF INDIA ZO KHANDWA          0.0       36.0       22.2        0.145              0
  BOI-SOBHAPUR BANK OF INDIA ZO KHANDWA         60.0       93.0       80.4        0.128            226
     BOI-BABAI BANK OF INDIA ZO KHANDWA         70.0      100.0       88.4        0.117            366
    BOI-GANERA BANK OF INDIA ZO KHANDWA         32.0       63.0       48.0        0.116              3
BOI-BALLYBAZAR BANK OF INDIA  ZO HOWRAH          0.0       27.0       17.0        0.102              0

🟢 TOP 10 RECOVERING DEVICES (best improvement):
Empty DataFrame
Columns: [device_name, bank_name, zo_name, score_start, sc


✅ Trend analysis sheet added to ts_summary_20260529_1559.xlsx
